In [ ]:
!apt-get update
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y ./google-chrome-stable_current_amd64.deb
!pip install selenium

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,971 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2026-04-14 18:32:27--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 74.12

# Scrapping Shops List

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
options.binary_location = "/usr/bin/google-chrome"

def initiate_driver():
    try:
        driver = webdriver.Chrome(options=options)
        print("Driver Works")
        return driver
    except Exception as e:
        print(f"Fail: {e}")
        return None

driver = initiate_driver()

Driver Works


In [ ]:
def get_driver():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    options.binary_location = "/usr/bin/google-chrome"
    return webdriver.Chrome(options=options)

pages_dict = {
    'a': 125, 'b': 110, 'c': 42, 'd': 29, 'e': 22,
    'f': 47, 'g': 85, 'h': 40, 'i': 77, 'j': 26,
    'k': 35, 'l': 29, 'm': 64, 'n': 21, 'o': 59,
    'p': 45, 'q': 3, 'r': 23, 's': 138, 't': 39,
    'u': 9, 'v': 12, 'w': 15, 'x': 3, 'y': 7,
    'z': 7}

In [ ]:
js_script = """
let dataToko = [];
let seenUrls = new Set();
let allLinks = document.querySelectorAll('a');
allLinks.forEach(el => {
    let text = el.innerText;
    let href = el.href;
    if (text.includes('Lihat Toko')) {
        let namaBersih = text.split('\\n')[0].trim();
        if (namaBersih && !seenUrls.has(href)) {
            dataToko.push({"Nama Toko": namaBersih, "URL": href});
            seenUrls.add(href);
        }
    }
});
return dataToko;
"""

list_shop = []

for query, max_page in pages_dict.items():
    print(f"Start query '{query}'...")
    driver = get_driver()
    empty_pages = 0
    toko_per_huruf = []
    try:
        for page in range(1, max_page + 1):
            url = f"https://www.tokopedia.com/search?page={page}&q={query}&shop_tier=2&st=shop"
            try:
                driver.get(url)
                time.sleep(5)
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
                time.sleep(2)
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(2)
                res = driver.execute_script(js_script)
                if res:
                    toko_per_huruf.extend(res)
                else:
                    empty_pages += 1
            except Exception as e:
                empty_pages += 1
                continue
            time.sleep(3)
    finally:
        driver.quit()
        list_shop.extend(toko_per_huruf)
        if empty_pages > 0:
            print(f"Query '{query}' is finished. Note: {empty_pages} pages were empty or failed.")
        else:
            print(f"Query '{query}' is finished successfully.")
        time.sleep(10)

Start query 'a'...
Query 'a' is finished. Note: 5 pages were empty or failed.
Start query 'b'...
Query 'b' is finished successfully.
Start query 'c'...
Query 'c' is finished successfully.
Start query 'd'...
Query 'd' is finished successfully.
Start query 'e'...
Query 'e' is finished successfully.
Start query 'f'...
Query 'f' is finished successfully.
Start query 'g'...
Query 'g' is finished successfully.
Start query 'h'...
Query 'h' is finished successfully.
Start query 'i'...
Query 'i' is finished successfully.
Start query 'j'...
Query 'j' is finished successfully.
Start query 'k'...
Query 'k' is finished successfully.
Start query 'l'...
Query 'l' is finished successfully.
Start query 'm'...
Query 'm' is finished successfully.
Start query 'n'...
Query 'n' is finished successfully.
Start query 'o'...
Query 'o' is finished successfully.
Start query 'p'...
Query 'p' is finished successfully.
Start query 'q'...
Query 'q' is finished successfully.
Start query 'r'...
Query 'r' is finished. 

KeyError: Index(['Url'], dtype='object')

In [ ]:
if list_shop:
    df = pd.DataFrame(list_shop).drop_duplicates(subset='Url')
    path_gdrive = "/content/drive/MyDrive/Scrapping_list_shop.csv"
    df.to_csv(path_gdrive, index=False)
else:
    print("No data was successfully retrieved.")

In [ ]:
list_shop

[{'Nama Toko': 'Awkward Brand',
  'URL': 'https://www.tokopedia.com/awkwardofficial'},
 {'Nama Toko': 'Apotek Sengeti Farma Sekernan',
  'URL': 'https://www.tokopedia.com/apoteksengetifarmasekern'},
 {'Nama Toko': 'AUTOPILOT FIX STORE',
  'URL': 'https://www.tokopedia.com/autopilotfix'},
 {'Nama Toko': 'Giordano Active Indonesia',
  'URL': 'https://www.tokopedia.com/thenorthface'},
 {'Nama Toko': 'Agnesis Mall',
  'URL': 'https://www.tokopedia.com/agnesisofficial'},
 {'Nama Toko': 'Dell Authorized Store Arena IT',
  'URL': 'https://www.tokopedia.com/dellarenait'},
 {'Nama Toko': 'KING AUTOCARE',
  'URL': 'https://www.tokopedia.com/king-autocare-'},
 {'Nama Toko': 'ARVOS Official',
  'URL': 'https://www.tokopedia.com/arvos-official'},
 {'Nama Toko': 'Apotek Sentra Salemba Senen',
  'URL': 'https://www.tokopedia.com/apoteksentrasalembasenen'},
 {'Nama Toko': 'Harigami Apparel',
  'URL': 'https://www.tokopedia.com/harigami'},
 {'Nama Toko': 'Apotek GMC Benda',
  'URL': 'https://www.tokope

In [ ]:
df = pd.DataFrame(list_shop).drop_duplicates(subset='URL')
path_gdrive = "/content/drive/MyDrive/Scrapping_list_shop.csv"
df.to_csv(path_gdrive, index=False)

In [ ]:
len(list_shop)

32291

In [ ]:
df.shape

(13165, 2)

# Scrapping Shop ID

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/Scrapping_list_shop.csv")
df

Mounted at /content/drive


,Nama Toko,URL
0,Awkward Brand,https://www.tokopedia.com/awkwardofficial
1,Apotek Sengeti Farma Sekernan,https://www.tokopedia.com/apoteksengetifarmase...
2,AUTOPILOT FIX STORE,https://www.tokopedia.com/autopilotfix
3,Giordano Active Indonesia,https://www.tokopedia.com/thenorthface
4,Agnesis Mall,https://www.tokopedia.com/agnesisofficial
...,...,...
13160,ZOLVYN,https://www.tokopedia.com/zolvyn
13161,Apotek Zahra Bekasi by GoApotik,https://www.tokopedia.com/apotek-zahra-bekasi-...
13162,Zalmore,https://www.tokopedia.com/zalmore
13163,Zwitsal,https://www.tokopedia.com/zwitsal-273


In [ ]:
import requests
import pandas as pd
import time

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36',
    'content-type': 'application/json',
    'accept': '*/*',
    'origin': 'https://www.tokopedia.com',
    'referer': 'https://www.tokopedia.com/',
    'x-version': 'cfd5462',
    'x-source': 'tokopedia-lite',
    'x-device': 'default_v3',
    'x-tkpd-lite-service': 'zeus',
}

def get_shop_sid(shop_idx, shop_url: str, retries: int = 5) -> str | None:
    domain = shop_url.rstrip('/').split('/')[-1]
    url = 'https://gql.tokopedia.com/graphql/ShopInfoByID'
    payload = [{
        "operationName": "ShopInfoByID",
        "variables": {"domain": domain, "id": 0},
        "query": """
        query ShopInfoByID($id: Int!, $domain: String) {
          shopInfoByID(input: {shopIDs: [$id], fields: ["core"], domain: $domain}) {
            result {
              shopCore { shopID name domain }
            }
            error { message }
          }
        }"""
    }]
    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=15)
            if resp.status_code == 200:
                result = resp.json()[0]['data']['shopInfoByID']['result']
                if result:
                    shop_core = result[0]['shopCore']
                    sid = str(shop_core['shopID'])
                    return sid
                else:
                    print(f"[{shop_idx}] No Result: {domain}")
                    return None
            elif resp.status_code == 502:
                wait = attempt * 2
                print(f"[{shop_idx}] Status {resp.status_code}: {domain} - {shop_url}  Retrying")
                print()
                time.sleep(wait)
            elif resp.status_code == 429:
                print(f"[{shop_idx}] 429 Rate limit {domain}  - {shop_url}")
                print()
                time.sleep(10)
            else:
                print(f"[{shop_idx}] Status {resp.status_code} untuk: {domain}")
                print()
                return None
        except requests.exceptions.Timeout:
            wait = attempt * 2
            print(f"[{shop_idx}] Timeout: {domain} - {shop_url}")
            time.sleep(wait)
        except Exception as e:
            print(f"[{shop_idx}] Error {domain} - {shop_url}: {e}")
            return None
    return None

def get_all_sids(df: pd.DataFrame,
                 shop_name: str = 'Nama Toko',
                 shop_url:  str = 'URL') -> pd.DataFrame:
    hasil = []
    total = len(df)
    for i, row in df.iterrows():
        nama = row[shop_name]
        url  = row[shop_url]
        sid = get_shop_sid(i, url)
        if not sid:
            print(f"[{i}] SID not Found on {nama}")
            print()
        hasil.append({
            shop_name:  nama,
            shop_url:   url,
            'SID':     sid,
        })
        time.sleep(0.3)
    df_hasil = pd.DataFrame(hasil)
    berhasil = df_hasil['SID'].notna().sum()
    gagal    = df_hasil['SID'].isna().sum()
    print(f"Success : {berhasil}")
    print(f"Fail    : {gagal}")
    return df_hasil

In [ ]:
df_sid = get_all_sids(
    df,
    shop_name = 'Nama Toko',
    shop_url  = 'URL'
)
print(df_sid)
df_sid.to_csv('toko_dengan_sid.csv', index=False)

[22] Error applesolution-store - https://www.tokopedia.com/applesolution-store: 'NoneType' object is not subscriptable
[22] SID not Found on APPLESOLUTION STORE

[23] Error kals-and-frey - https://www.tokopedia.com/kals-and-frey: 'NoneType' object is not subscriptable
[23] SID not Found on Kals and Frey

[25] Error asfindo-scents - https://www.tokopedia.com/asfindo-scents: 'NoneType' object is not subscriptable
[25] SID not Found on Asfindo Scents

[26] Error airnderm-by-airin-beauty - https://www.tokopedia.com/airnderm-by-airin-beauty: 'NoneType' object is not subscriptable
[26] SID not Found on Airnderm by Airin Beauty

[27] Error amazeus-914 - https://www.tokopedia.com/amazeus-914: 'NoneType' object is not subscriptable
[27] SID not Found on amazeus

[32] Error arummi-cashew-milk-official - https://www.tokopedia.com/arummi-cashew-milk-official: 'NoneType' object is not subscriptable
[32] SID not Found on Arummi Cashew Milk Official

[35] Error anmo-yoga - https://www.tokopedia.com/a

In [ ]:
df_sid

,Nama Toko,URL,SID
0,Awkward Brand,https://www.tokopedia.com/awkwardofficial,14656547
1,Apotek Sengeti Farma Sekernan,https://www.tokopedia.com/apoteksengetifarmase...,17115861
2,AUTOPILOT FIX STORE,https://www.tokopedia.com/autopilotfix,4779108
3,Giordano Active Indonesia,https://www.tokopedia.com/thenorthface,12224845
4,Agnesis Mall,https://www.tokopedia.com/agnesisofficial,9614821
...,...,...,...
13160,ZOLVYN,https://www.tokopedia.com/zolvyn,7495483106462304909
13161,Apotek Zahra Bekasi by GoApotik,https://www.tokopedia.com/apotek-zahra-bekasi-...,17637111
13162,Zalmore,https://www.tokopedia.com/zalmore,6583228
13163,Zwitsal,https://www.tokopedia.com/zwitsal-273,7494925843634358389


In [ ]:
df_sid.isna().sum()

,0
Nama Toko,0
URL,0
SID,268


In [ ]:
df_sid[df_sid["SID"].isna()]

,Nama Toko,URL,SID
22,APPLESOLUTION STORE,https://www.tokopedia.com/applesolution-store,None
23,Kals and Frey,https://www.tokopedia.com/kals-and-frey,None
25,Asfindo Scents,https://www.tokopedia.com/asfindo-scents,None
26,Airnderm by Airin Beauty,https://www.tokopedia.com/airnderm-by-airin-be...,None
27,amazeus,https://www.tokopedia.com/amazeus-914,None
...,...,...,...
13067,Yesplus,https://www.tokopedia.com/yesplus-673,None
13113,Zode,https://www.tokopedia.com/zode,None
13114,Zestmag_NEW,https://www.tokopedia.com/zestmag,None
13115,ZarsOfficialStore,https://www.tokopedia.com/zarsofficialstore,None


In [ ]:
df_sid = df_sid.dropna().reset_index(drop=True)

In [ ]:
path_gdrive = "/content/drive/MyDrive/Scrapping_list_shop_with_SID.csv"
df_sid.to_csv(path_gdrive, index=False)

In [ ]:
df_sid

,Nama Toko,URL,SID
0,Awkward Brand,https://www.tokopedia.com/awkwardofficial,14656547
1,Apotek Sengeti Farma Sekernan,https://www.tokopedia.com/apoteksengetifarmase...,17115861
2,AUTOPILOT FIX STORE,https://www.tokopedia.com/autopilotfix,4779108
3,Giordano Active Indonesia,https://www.tokopedia.com/thenorthface,12224845
4,Agnesis Mall,https://www.tokopedia.com/agnesisofficial,9614821
...,...,...,...
12892,ZOLVYN,https://www.tokopedia.com/zolvyn,7495483106462304909
12893,Apotek Zahra Bekasi by GoApotik,https://www.tokopedia.com/apotek-zahra-bekasi-...,17637111
12894,Zalmore,https://www.tokopedia.com/zalmore,6583228
12895,Zwitsal,https://www.tokopedia.com/zwitsal-273,7494925843634358389


# Scrapping Products (Price, Rating, Product_Url, Image_Url

## Preparing

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/Scrapping_list_shop_with_SID.csv")
df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Nama Toko,URL,SID
0,Awkward Brand,https://www.tokopedia.com/awkwardofficial,14656547
1,Apotek Sengeti Farma Sekernan,https://www.tokopedia.com/apoteksengetifarmase...,17115861
2,AUTOPILOT FIX STORE,https://www.tokopedia.com/autopilotfix,4779108
3,Giordano Active Indonesia,https://www.tokopedia.com/thenorthface,12224845
4,Agnesis Mall,https://www.tokopedia.com/agnesisofficial,9614821
...,...,...,...
12892,ZOLVYN,https://www.tokopedia.com/zolvyn,7495483106462304909
12893,Apotek Zahra Bekasi by GoApotik,https://www.tokopedia.com/apotek-zahra-bekasi-...,17637111
12894,Zalmore,https://www.tokopedia.com/zalmore,6583228
12895,Zwitsal,https://www.tokopedia.com/zwitsal-273,7494925843634358389


In [ ]:
import requests
import pandas as pd
import time
import sqlite3
import os
def scrape_all_product(df_shops, db_name="tokopedia_data.db"):
    folder_name = 'Gambar'
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    url = 'https://gql.tokopedia.com/graphql/ShopProducts'
    headers = {
        'sec-ch-ua-platform': '"Windows"',
        'x-version': 'cfd5462',
        'Referer': 'https://www.tokopedia.com/',
        'sec-ch-ua': '"Google Chrome";v="147", "Not.A/Brand";v="8", "Chromium";v="147"',
        'x-price-center': 'true',
        'sec-ch-ua-mobile': '?0',
        'bd-device-id': '7628337271406577160',
        'x-source': 'tokopedia-lite',
        'x-device': 'default_v3',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36',
        'accept': '*/*',
        'content-type': 'application/json',
        'x-tkpd-lite-service': 'zeus'
    }
    all_extracted_products = []
    error_list = []
    image_error_list = []
    global_id = 1
    try:
        for index, row in df_shops.iterrows():
            nama_toko = row['Nama Toko']
            sid = str(row['SID'])
            page = 1
            while True:
                payload = [{
                    "operationName": "ShopProducts",
                    "variables": {
                        "source": "shop", "sid": sid, "page": page, "perPage": 80,
                        "etalaseId": "etalase", "sort": 1, "user_districtId": "2274",
                        "user_cityId": "176", "user_lat": "0", "user_long": "0",
                        "usecase": "ace_get_shop_product_v2"
                    },
                    "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    status\n    errors\n    links {\n      prev\n      next\n      __typename\n    }\n    data {\n      name\n      product_url\n      product_id\n      price {\n        text_idr\n        __typename\n      }\n      primary_image {\n        original\n        thumbnail\n        resize300\n        __typename\n      }\n      flags {\n        isSold\n        isPreorder\n        isWholesale\n        isWishlist\n        __typename\n      }\n      campaign {\n        discounted_percentage\n        original_price_fmt\n        start_date\n        end_date\n        __typename\n      }\n      label {\n        color_hex\n        content\n        __typename\n      }\n      label_groups {\n        position\n        title\n        type\n        url\n        styles {\n          key\n          value\n          __typename\n        }\n        __typename\n      }\n      badge {\n        title\n        image_url\n        __typename\n      }\n      stats {\n        reviewCount\n        rating\n        averageRating\n        __typename\n      }\n      category {\n        id\n        __typename\n      }\n      __typename\n    }\n    __typename\n  }\n}\n"
                }]
                try:
                    response = requests.post(url, headers=headers, json=payload, timeout=15)
                    if response.status_code != 200:
                        error_list.append({'Index_DF': index, 'Nama Toko': nama_toko, 'SID': sid, 'Pesan_Error': f'HTTP {response.status_code}'})
                        break
                    raw = response.json()
                    if not raw or 'data' not in raw[0] or 'GetShopProduct' not in raw[0]['data'] or 'data' not in raw[0]['data']['GetShopProduct']:
                        error_list.append({'Index_DF': index, 'Nama Toko': nama_toko, 'SID': sid, 'Pesan_Error': 'Struktur JSON tidak valid (Mungkin diblokir)'})
                        break
                    products = raw[0]['data']['GetShopProduct']['data']
                    if not products or len(products) == 0:
                        break
                    for p in products:
                        rating = p.get('stats', {}).get('averageRating', "0")
                        all_extracted_products.append({
                            'Id': global_id,
                            'Shop_Name': nama_toko,
                            'Product_Name': p.get('name', ''),
                            'Price': p.get('price', {}).get('text_idr', ''),
                            'Rating': rating,
                            'URL_Product_Picture': p.get('primary_image', {}).get('original', ''),
                            'URL_Product': p.get('product_url', '')
                        })
                        url_gambar = p.get('primary_image', {}).get('original', '')
                        if url_gambar:
                            try:
                                img_response = requests.get(url_gambar, headers={'User-Agent': headers['User-Agent']}, timeout=10)
                                if img_response.status_code == 200:
                                    save_path = os.path.join(folder_name, f"{global_id}.jpg")
                                    with open(save_path, 'wb') as f:
                                        f.write(img_response.content)
                                else:
                                    image_error_list.append({'Index_Product': global_id, 'Nama Toko': nama_toko, 'SID': sid,'Product_Name': p.get('name', '')})
                            except Exception:
                                image_error_list.append({'Index_Product': global_id, 'Nama Toko': nama_toko, 'SID': sid,'Product_Name': p.get('name', '')})
                        global_id += 1
                    page += 1
                    time.sleep(1.5)
                except Exception as e:
                    error_list.append({'Index_DF': index, 'Nama Toko': nama_toko, 'SID': sid, 'Pesan_Error': f'Exception: {str(e)}'})
                    break
    except KeyboardInterrupt:
        print("\nScraping process terminated by user (KeyboardInterrupt).")
        print("Saving collected data to the database before exiting...")
    if all_extracted_products:
        df_final = pd.DataFrame(all_extracted_products)
        conn = sqlite3.connect(db_name)
        df_final.to_sql('tokopedia_product', conn, if_exists='replace', index=False)
        conn.close()
        print(f"Successfully saved {len(df_final)} products to {db_name}.")
        print(f"Encountered {len(error_list)} errors during the process.")
        print(f"Total fail images: {image_error_list}")
        return df_final, error_list, image_error_list
    else:
        print("\nNo product data was successfully retrieved.")
        return None, error_list

## Phase 1

In [ ]:
df_phase1 = df.iloc[:2000]
df_phase1.tail(5)

,Nama Toko,URL,SID
1995,Apotek Fegas Farma By GoApotik,https://www.tokopedia.com/apotek-fegas-farma-b...,16684340
1996,Apotek Taruma Sehat Jakarta Barat by GoApotik,https://www.tokopedia.com/apotek-taruma-sehat-...,17871496
1997,Apotek Ratu Medika Sukabumi,https://www.tokopedia.com/apotekratumedikas,12785194
1998,Apotek Hilda Alnaira Jakarta Utara by GoApotik,https://www.tokopedia.com/apotek-hilda-alnaira...,13965466
1999,Apotek Suka Maju Mencirim by GoApotik,https://www.tokopedia.com/apotek-suka-maju-men...,17030811


In [ ]:
df_phase1, list_error_phase1,list_error_gambar_phase1 = scrape_all_product(df_phase1, db_name="database_products_phase1.db")

In [ ]:
print()

In [ ]:
df_phase1

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,1,Awkward Brand,Kaos Polos Anak Katun Combed 30s Hijau Olive,Rp74.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
1,2,Awkward Brand,Kaos Polos Anak Katun Combed 30s Biru Navy,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
2,3,Awkward Brand,Kaos Polos Anak Katun Combed 30s Kuning Mustard,Rp74.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
3,4,Awkward Brand,Kaos Polos Anak Katun Combed 30s Merah Maroon,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
4,5,Awkward Brand,Kaos Polos Anak Katun Combed 30s Abu-abu Misty,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
...,...,...,...,...,...,...,...
388,389,Agnesis Mall,Agnesis Premium Elbow Band / Pelindung Siku,Rp34.425,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/agnesisofficial/agne...
389,390,Agnesis Mall,Agnesis Man Swimsuit / Celana Renang Pria Agnesis,Rp93.600,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/agnesisofficial/agne...
390,391,Agnesis Mall,Agnesis Chess (Catur) Set Deluxe / Papan Catur...,Rp138.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/agnesisofficial/agne...
391,392,Agnesis Mall,Agnesis Brief Supporter / Celana Dalam Hernia ...,Rp65.565,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/agnesisofficial/agne...


In [ ]:
df_phase1, list_error_phase1 = scrape_all_product(df_phase1, db_name="database_products_phase1.db")

Successfully saved 1161234 products to database_products_phase1.db.
Encountered 20 errors during the process.


In [ ]:
df_phase1.to_csv(f'/content/drive/MyDrive/df_phase1.csv', index=False)
df_error_phase_1 = pd.DataFrame(list_error_phase1)
df_error_phase_1.to_csv(f'/content/drive/MyDrive/df_error_phase_1.csv', index=False)

## Phase 2

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
df_error_phase_1 = pd.read_csv("/content/drive/MyDrive/df_error_phase_1.csv")
df_error_phase_1

Mounted at /content/drive


,Index_DF,Nama Toko,SID,Pesan_Error
0,442,ALEGRO99.,7494889111453338242,Exception: HTTPSConnectionPool(host='gql.tokop...
1,444,Amarta Herbal Store,7494934202377276227,Exception: HTTPSConnectionPool(host='gql.tokop...
2,446,Apotek Alkafi Farma Mojokerto by GoApotik,16854450,Exception: HTTPSConnectionPool(host='gql.tokop...
3,447,Apotek Alpro Express Taqwa,11554476,Exception: HTTPSConnectionPool(host='gql.tokop...
4,448,Apotek Kita Bekasi,14702582,Exception: HTTPSConnectionPool(host='gql.tokop...
5,592,Apotek Kreator Kesehatan Cipondoh,14336143,Exception: HTTPSConnectionPool(host='gql.tokop...
6,696,Apotek Azzahra 24 by GoApotik,18029821,Exception: HTTPSConnectionPool(host='gql.tokop...
7,697,Apotek Athar Farma Palembang by GoApotik,17224885,Exception: HTTPSConnectionPool(host='gql.tokop...
8,700,Apotek Nayra Cirebon by GoApotik,17871515,Exception: HTTPSConnectionPool(host='gql.tokop...
9,701,AL Leather,13268421,Exception: HTTPSConnectionPool(host='gql.tokop...


In [ ]:
df_phase_1 = pd.read_csv("/content/drive/MyDrive/df_phase1.csv")
df_phase_1.tail(5)

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
1161229,1161230,PKB Bandung,VIVA ASTRINGENT CUCUMBER 200 ML Toner Wajah,Rp12.927,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/pkb-bandung/viva-ast...
1161230,1161231,PKB Bandung,Viva Anti Winkel Cream 22 gr,Rp13.857,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/pkb-bandung/viva-ant...
1161231,1161232,PKB Bandung,VIVA AIR MAWAR 200 ML | ROSE WATER Bunga Peny...,Rp12.071,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/pkb-bandung/viva-air...
1161232,1161233,PKB Bandung,Viva Face Tonic Green Tea 100 ml Berjerawat P...,Rp9.114,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/pkb-bandung/viva-fac...
1161233,1161234,PKB Bandung,Viva White Waterdrop Sleeping Mask 80 gr Extra...,Rp29.342,4.9,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/pkb-bandung/viva-whi...


In [ ]:
df_continue_error_phase1 = df.loc[df_error_phase_1["Index_DF"]]

In [ ]:
df_continue_error_phase1

,Nama Toko,URL,SID
442,ALEGRO99.,https://www.tokopedia.com/alegro99-251,7494889111453338242
444,Amarta Herbal Store,https://www.tokopedia.com/amarta-herbal-store,7494934202377276227
446,Apotek Alkafi Farma Mojokerto by GoApotik,https://www.tokopedia.com/apotek-alkafi-farma-...,16854450
447,Apotek Alpro Express Taqwa,https://www.tokopedia.com/apotekgeneriktaqwa,11554476
448,Apotek Kita Bekasi,https://www.tokopedia.com/apotekkitabekasi,14702582
592,Apotek Kreator Kesehatan Cipondoh,https://www.tokopedia.com/apotekkreatorkesehat...,14336143
696,Apotek Azzahra 24 by GoApotik,https://www.tokopedia.com/apotek-azzahra-24-by...,18029821
697,Apotek Athar Farma Palembang by GoApotik,https://www.tokopedia.com/apotek-athar-farma-p...,17224885
700,Apotek Nayra Cirebon by GoApotik,https://www.tokopedia.com/apotek-nayra-cirebon...,17871515
701,AL Leather,https://www.tokopedia.com/al-leather,13268421


In [ ]:
df_phase2, list_error_phase2 = scrape_all_product(df_continue_error_phase1, db_name="database_products_phase1.db")

Successfully saved 16424 products to database_products_phase1.db.
Encountered 0 errors during the process.


In [ ]:
df_phase2.to_csv(f'/content/drive/MyDrive/df_phase2.csv', index=False)

In [ ]:
df_phase2.head(5)

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,1161235,ALEGRO99.,sweater Hoodie DEDEMIT metal termurah Distrok...,Rp127.500,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/alegro99-251/sweater...
1,1161236,ALEGRO99.,Jaket Sweater ALEGRO99 JPG Hodie Polos Ori Pri...,Rp230.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/alegro99-251/jaket-s...
2,1161237,ALEGRO99.,Alegro99 Keep Quit Celana Pendek Jumbo Cargo P...,Rp45.000,,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/alegro99-251/alegro9...
3,1161238,ALEGRO99.,Alegro99 Spider Celana Pendek Jumbo Cargo Pri...,Rp45.000,,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/alegro99-251/alegro9...
4,1161239,ALEGRO99.,Alegro99 Stay Smoke Celana Pendek Jumbo Cargo...,Rp45.000,,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/alegro99-251/alegro9...


## Phase 3

In [ ]:
df_phase2.tail(5)

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
16419,1177654,Apotek Golden Sehat by GoApotik,FIESTA KONDOM STRAWBERRY BOX 3 PCS,Rp16.755,,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-golden-sehat-...
16420,1177655,Apotek Golden Sehat by GoApotik,DUMIN SYRUP ISI 60 ML BOTOL,Rp42.081,,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-golden-sehat-...
16421,1177656,Apotek Golden Sehat by GoApotik,DAKTARIN DIAPERS SALEP 10 GRAM,Rp71.694,,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-golden-sehat-...
16422,1177657,Apotek Golden Sehat by GoApotik,DUMIN 250 MG RECTAL TUBE 4 ML,Rp41.433,,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-golden-sehat-...
16423,1177658,Apotek Golden Sehat by GoApotik,CEREBROVIT X CELL STRIP 10 KAPSUL,Rp2.599,,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-golden-sehat-...


In [ ]:
df_phase3 = df.iloc[3224:6449]
df_phase3.head(5)

,Nama Toko,URL,SID
3224,By.A Herbal,https://www.tokopedia.com/bya-herbal,7495160669446113764
3225,Cerita Bandung Official,https://www.tokopedia.com/cerita-bandung-official,5098392
3226,BrothersNetworks,https://www.tokopedia.com/brothersnetwork,9341152
3227,Boostrike,https://www.tokopedia.com/boostrike,7494803684608804872
3228,Tiramisusu by Chocomory,https://www.tokopedia.com/tiramisusu,8299817


In [ ]:
df_phase3, list_error_phase3 = scrape_all_product(df_phase3, db_name="database_products_phase1.db")

Successfully saved 1016204 products to database_products_phase1.db.
Encountered 17 errors during the process.


In [ ]:
df_phase3.to_csv(f'/content/drive/MyDrive/df_phase3.csv', index=False)
df_error_phase_3 = pd.DataFrame(list_error_phase3)
df_error_phase_3.to_csv(f'/content/drive/MyDrive/df_error_phase_3.csv', index=False)

In [ ]:
df_phase3

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,1177659,By.A Herbal,Sendifit Asli Obat Asam Urat Dan Rematik (100%...,Rp137.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/sendifit-...
1,1177660,By.A Herbal,Cara Mengobati Asam Lambung Dengan Kapsul Seha...,Rp125.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/cara-meng...
2,1177661,By.A Herbal,hanya packing herbal keloreena,Rp999.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/hanya-pac...
3,1177662,By.A Herbal,"Obat Sesak Di Dada , Nafas Berat, Nyeri Ulu Ha...",Rp125.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/obat-sesa...
4,1177663,By.A Herbal,Kopi Herbal Radimax Isi 10 Sachet Sudah BPOM &...,Rp150.000,4.3,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/kopi-herb...
...,...,...,...,...,...,...,...
1016199,2193858,Fimory Mall,Beli dengan voucher Fimory Gastro Original Pak...,Rp285.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/beli-den...
1016200,2193859,Fimory Mall,Fimory Gastro Original Paket 5 Box - Minuman C...,Rp475.000,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...
1016201,2193860,Fimory Mall,Fimory Gastro Best Seller Paket 2 Box - Cereal...,Rp194.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...
1016202,2193861,Fimory Mall,Fimory Gastro Original Paket 6 Box - Minuman S...,Rp570.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...


In [ ]:
df_error_phase_3

,Index_DF,Nama Toko,SID,Pesan_Error
0,3233,Mizzu Bottle Official,7496199787596843126,Exception: HTTPSConnectionPool(host='gql.tokop...
1,3347,Hypermart Balekota,10178602,Exception: HTTPSConnectionPool(host='gql.tokop...
2,3349,Hyfresh Bojongsari,12468295,Exception: HTTPSConnectionPool(host='gql.tokop...
3,3352,Bakpia Tugu Jogja,8308263,Exception: HTTPSConnectionPool(host='gql.tokop...
4,3511,Agres ID Bandung Timur,7496302951648955384,Exception: HTTPSConnectionPool(host='gql.tokop...
5,3540,Apotek Aiman by GoApotik,13147693,Exception: HTTPSConnectionPool(host='gql.tokop...
6,3817,Berkah Saluyu,4008433,Exception: HTTPSConnectionPool(host='gql.tokop...
7,3851,Bintang Official Store,7495098429600860861,Exception: HTTPSConnectionPool(host='gql.tokop...
8,3944,bss911,7496183526509546410,Exception: HTTPSConnectionPool(host='gql.tokop...
9,4370,Apotek Yenni by GoApotik,13805660,Exception: HTTPSConnectionPool(host='gql.tokop...


## phase 4

In [ ]:
drive.mount('/content/drive')
df_error_phase_3 = pd.read_csv("/content/drive/MyDrive/df_error_phase_3.csv")
df_error_phase_3

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Index_DF,Nama Toko,SID,Pesan_Error
0,3233,Mizzu Bottle Official,7496199787596843126,Exception: HTTPSConnectionPool(host='gql.tokop...
1,3347,Hypermart Balekota,10178602,Exception: HTTPSConnectionPool(host='gql.tokop...
2,3349,Hyfresh Bojongsari,12468295,Exception: HTTPSConnectionPool(host='gql.tokop...
3,3352,Bakpia Tugu Jogja,8308263,Exception: HTTPSConnectionPool(host='gql.tokop...
4,3511,Agres ID Bandung Timur,7496302951648955384,Exception: HTTPSConnectionPool(host='gql.tokop...
5,3540,Apotek Aiman by GoApotik,13147693,Exception: HTTPSConnectionPool(host='gql.tokop...
6,3817,Berkah Saluyu,4008433,Exception: HTTPSConnectionPool(host='gql.tokop...
7,3851,Bintang Official Store,7495098429600860861,Exception: HTTPSConnectionPool(host='gql.tokop...
8,3944,bss911,7496183526509546410,Exception: HTTPSConnectionPool(host='gql.tokop...
9,4370,Apotek Yenni by GoApotik,13805660,Exception: HTTPSConnectionPool(host='gql.tokop...


In [ ]:
df_phase_3 = pd.read_csv("/content/drive/MyDrive/df_phase3.csv")
df_phase_3.tail(5)

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
1016199,2193858,Fimory Mall,Beli dengan voucher Fimory Gastro Original Pak...,Rp285.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/beli-den...
1016200,2193859,Fimory Mall,Fimory Gastro Original Paket 5 Box - Minuman C...,Rp475.000,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...
1016201,2193860,Fimory Mall,Fimory Gastro Best Seller Paket 2 Box - Cereal...,Rp194.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...
1016202,2193861,Fimory Mall,Fimory Gastro Original Paket 6 Box - Minuman S...,Rp570.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...
1016203,2193862,Fimory Mall,Fimory Gastro Original Paket 4 Box - Minuman C...,Rp380.000,4.9,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...


In [ ]:
df_continue_error_phase3 = df.loc[df_error_phase_3["Index_DF"]]
df_continue_error_phase3

,Nama Toko,URL,SID
3233,Mizzu Bottle Official,https://www.tokopedia.com/mizzu-bottle-official,7496199787596843126
3347,Hypermart Balekota,https://www.tokopedia.com/hpmbalekota,10178602
3349,Hyfresh Bojongsari,https://www.tokopedia.com/hyfresh-bojongsari,12468295
3352,Bakpia Tugu Jogja,https://www.tokopedia.com/bakpiakukustugujogja-,8308263
3511,Agres ID Bandung Timur,https://www.tokopedia.com/agres-id-bandung-timur,7496302951648955384
3540,Apotek Aiman by GoApotik,https://www.tokopedia.com/apotek-aiman-by-goap...,13147693
3817,Berkah Saluyu,https://www.tokopedia.com/berkahsaluyu,4008433
3851,Bintang Official Store,https://www.tokopedia.com/bintang-official-store,7495098429600860861
3944,bss911,https://www.tokopedia.com/bss911,7496183526509546410
4370,Apotek Yenni by GoApotik,https://www.tokopedia.com/apotek-yenni-by-goap...,13805660


In [ ]:
df_phase4, list_error_phase4 = scrape_all_product(df_continue_error_phase3, db_name="database_products_phase1.db")

Successfully saved 23363 products to database_products_phase1.db.
Encountered 0 errors during the process.


In [ ]:
df_phase4

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,2193863,Mizzu Bottle Official,Mizzu Bottle - Brew Botol Minum Stainless Cera...,Rp212.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mizzu-bottle-officia...
1,2193864,Mizzu Bottle Official,Mizzu Bottle - Flow Botol Minum Stainless Tumb...,Rp210.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mizzu-bottle-officia...
2,2193865,Mizzu Bottle Official,Mizzu Bottle - Barrel Botol Minum Stainless Tu...,Rp267.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mizzu-bottle-officia...
3,2193866,Mizzu Bottle Official,Mizzu Bottle - Urban Tumbler Botol Minum Stain...,Rp145.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mizzu-bottle-officia...
4,2193867,Mizzu Bottle Official,Mizzu Grip Tumbler Besar Botol Minum Stainless...,Rp159.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mizzu-bottle-officia...
...,...,...,...,...,...,...,...
23358,2217221,Happy Family Storee,(HF) LOKAL Keset Lantai Cushion Printing Anti ...,Rp25.200,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/happyfamilyofficial/...
23359,2217222,Happy Family Storee,(HF) LOKAL Keset Lantai Glossy Nyerap Air Moti...,Rp21.100,4.6,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/happyfamilyofficial/...
23360,2217223,Happy Family Storee,(HF) LOKAL Keset Lantai MOTIF Modern UK 40x60...,Rp19.600,4.8,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/happyfamilyofficial/...
23361,2217224,Happy Family Storee,(HF) SNI Playmat Lipat Anak XPE Matras Karpet ...,Rp106.000,4.8,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/happyfamilyofficial/...


In [ ]:
df_phase4.to_csv(f'/content/drive/MyDrive/df_phase4.csv', index=False)

## Phase 5

In [ ]:
df_phase5 = df.iloc[6449:9673]
df_phase5.head(5)

,Nama Toko,URL,SID
6449,Weimar Factory,https://www.tokopedia.com/weimarofficial,8300785
6450,FibreFirst,https://www.tokopedia.com/fibrefirst,4040580
6451,YesaMalika Fashion,https://www.tokopedia.com/yesamalika-fashion,7494455353818516385
6452,MOCO Fashion,https://www.tokopedia.com/moco-fashion,7494951145258126309
6453,FLATE_NEW,https://www.tokopedia.com/flate,7496026743576234202


In [ ]:
df_phase5, list_error_phase5 = scrape_all_product(df_phase5, db_name="database_products_phase1.db")

Successfully saved 775750 products to database_products_phase1.db.
Encountered 0 errors during the process.


In [ ]:
df_phase5

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,2217226,Weimar Factory,Njonjamar - Kue Kacang Wijen Kucang Jadul Kue ...,Rp 30.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://tokopedia.com/weimarofficial/njonjamar...
1,2217227,Weimar Factory,Binbon Stick Gemoy - Stik Gemoyy Coklat @250 G...,Rp 25.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://tokopedia.com/weimarofficial/binbon-st...
2,2217228,Weimar Factory,Binbon Stick Gemoy - Stik Gemoy KEJU @250 Gr R...,Rp 25.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://tokopedia.com/weimarofficial/binbon-st...
3,2217229,Weimar Factory,Weimar x Luvme Choco Jelly Fruit Jam - Coklat ...,Rp 50.000,,https://p19-images-sign-sg.tokopedia-static.ne...,https://tokopedia.com/weimarofficial/weimar-x-...
4,2217230,Weimar Factory,Binbon Jet Kentang - Rasa Original @200 Gr Sna...,Rp 15.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://tokopedia.com/weimarofficial/binbon-je...
...,...,...,...,...,...,...,...
775745,2992971,LJS Phone,Apple iPhone 17 256 GB 512GB Garansi Resmi In...,Rp17.399.000,5.0,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ljsofficial/apple-ip...
775746,2992972,LJS Phone,tempered glass redmi note 8,Rp35.000,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ljsofficial/tempered...
775747,2992973,LJS Phone,Tempered Glass Xiaomi Redmi 6A Full 5D,Rp50.000,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ljsofficial/tempered...
775748,2992974,LJS Phone,XTRA BUBBLE WRAP,Rp5.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ljsofficial/xtra-bub...


In [ ]:
df_phase5.to_csv(f'/content/drive/MyDrive/df_phase5.csv', index=False)

## phase 6

In [ ]:
df_phase6 = df.iloc[9673:]
df_phase6.head(5)

,Nama Toko,URL,SID
9673,Lenovo Legion Official,https://www.tokopedia.com/lenovolegion,9795705
9674,LABBANG,https://www.tokopedia.com/labbang-234,7495815086823541753
9675,Licensee SSD Store,https://www.tokopedia.com/hpstorage,6995044
9676,Lenovo Official,https://www.tokopedia.com/lenovo-official,2411241
9677,LUNAHABIT LOGO,https://www.tokopedia.com/lunahabit-logo,7494251967220516018


In [ ]:
df_phase6, list_error_phase6 = scrape_all_product(df_phase6, db_name="database_products_phase1.db")

Successfully saved 636154 products to database_products_phase1.db.
Encountered 9 errors during the process.


In [ ]:
df_phase6.to_csv(f'/content/drive/MyDrive/df_phase6.csv', index=False)

## Phase 7

In [ ]:
df_error_phase_7 = pd.DataFrame(list_error_phase6)
df_continue_error_phase7 = df.loc[df_error_phase_7["Index_DF"]]
df_continue_error_phase7

,Nama Toko,URL,SID
9753,Laviola Shoes,https://www.tokopedia.com/laviolashoes,2014745
10583,Menta Toys,https://www.tokopedia.com/menta-toys,7496151467763730440
10633,NFV SUNDANESIA,https://www.tokopedia.com/nfv-sundanesia,429064
11113,WELLEN PRINT OFFICIAL,https://www.tokopedia.com/wellenprint,5446738
12244,SIM XPRESS,https://www.tokopedia.com/sim-xpress,7494182158479427406
12393,tokostd_NEW,https://www.tokopedia.com/tokostd-854,9776891
12456,Tennesy,https://www.tokopedia.com/tennesyhomedecoration,12859077
12561,UniGet,https://www.tokopedia.com/uniget,7496201250891860195
12572,Apotek Ubay Farma by GoApotik,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250


In [ ]:
df_phase6.tail(5)

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
636149,3629125,Zoya,Zoya HARDBOX Scarf - Box Hampers Scarf [BOX SAJA],Rp25.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/zoya-hardbox-sc...
636150,3629126,Zoya,Kerudung Hijab Instan Bergo - Zoya Queena Berg...,Rp79.200,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/kerudung-hijab-...
636151,3629127,Zoya,Zoya - Kamaniya Scarf - Kerudung Hijab Segiemp...,Rp44.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/zoya-kamaniya-s...
636152,3629128,Zoya,Zoya Bergo Hijab Instant Marsha Glittering Cas...,Rp107.100,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/zoya-bergo-mars...
636153,3629129,Zoya,[BEST SELLER] Zoya Bergo Instan Marsha HL | Hi...,Rp71.100,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/zoya-bergo-hija...


In [ ]:
df_phase7, list_error_phase7 = scrape_all_product(df_continue_error_phase7, db_name="database_products_phase1.db")

Successfully saved 10203 products to database_products_phase1.db.
Encountered 0 errors during the process.


In [ ]:
df_phase7

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,3629130,Laviola Shoes,Laviola 5299 LSH - Sepatu Flat Wanita Beige da...,Rp215.910,,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/laviolashoes/laviola...
1,3629131,Laviola Shoes,Laviola 5298 LSH - Sepatu Flat Wanita BLACK da...,Rp215.910,,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/laviolashoes/laviola...
2,3629132,Laviola Shoes,Laviola 5293 LSH - Sepatu Flat Wanita Black Be...,Rp215.910,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/laviolashoes/laviola...
3,3629133,Laviola Shoes,Laviola 5292 LSH - Sepatu Flat Wanita Black Ap...,Rp215.910,,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/laviolashoes/laviola...
4,3629134,Laviola Shoes,Laviola 5239 LSF - Sepatu Flat Wanita Black da...,Rp215.910,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/laviolashoes/laviola...
...,...,...,...,...,...,...,...
10198,3639328,Apotek Ubay Farma by GoApotik,COMBIVENT UDV CAIRAN INHALASI 2.5 ML PACK 10 VIAL,Rp66.526,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
10199,3639329,Apotek Ubay Farma by GoApotik,COMBIVENT UDV CAIRAN INHALASI 2.5 ML BOX 20 VIAL,Rp114.908,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
10200,3639330,Apotek Ubay Farma by GoApotik,FLUIMUCIL 100 MG/ML CAIRAN INHALASI 3 ML AMPUL,Rp23.138,5.0,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
10201,3639331,Apotek Ubay Farma by GoApotik,FLUIMUCIL 100 MG/ML CAIRAN INHALASI 3 ML BOX 5...,Rp109.595,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...


In [ ]:
df_phase7.to_csv(f'/content/drive/MyDrive/df_phase7.csv', index=False)

## Merging all data

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
df1 = pd.read_csv('/content/drive/MyDrive/df_phase1.csv')
df2 = pd.read_csv('/content/drive/MyDrive/df_phase2.csv')
df3 = pd.read_csv('/content/drive/MyDrive/df_phase3.csv')
df4 = pd.read_csv('/content/drive/MyDrive/df_phase4.csv')
df5 = pd.read_csv('/content/drive/MyDrive/df_phase5.csv')
df6 = pd.read_csv('/content/drive/MyDrive/df_phase6.csv')
df7 = pd.read_csv('/content/drive/MyDrive/df_phase7.csv')

Mounted at /content/drive


Identifying and removing duplicate shop names during the Phase 1 scraping process.

In [ ]:
df1

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,1,Awkward Brand,Kaos Polos Anak Katun Combed 30s Hijau Olive,Rp74.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
1,2,Awkward Brand,Kaos Polos Anak Katun Combed 30s Biru Navy,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
2,3,Awkward Brand,Kaos Polos Anak Katun Combed 30s Kuning Mustard,Rp74.000,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
3,4,Awkward Brand,Kaos Polos Anak Katun Combed 30s Merah Maroon,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
4,5,Awkward Brand,Kaos Polos Anak Katun Combed 30s Abu-abu Misty,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
...,...,...,...,...,...,...,...
1161229,1161230,PKB Bandung,VIVA ASTRINGENT CUCUMBER 200 ML Toner Wajah,Rp12.927,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/pkb-bandung/viva-ast...
1161230,1161231,PKB Bandung,Viva Anti Winkel Cream 22 gr,Rp13.857,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/pkb-bandung/viva-ant...
1161231,1161232,PKB Bandung,VIVA AIR MAWAR 200 ML | ROSE WATER Bunga Peny...,Rp12.071,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/pkb-bandung/viva-air...
1161232,1161233,PKB Bandung,Viva Face Tonic Green Tea 100 ml Berjerawat P...,Rp9.114,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/pkb-bandung/viva-fac...


In [ ]:
df2

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,1161235,ALEGRO99.,sweater Hoodie DEDEMIT metal termurah Distrok...,Rp127.500,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/alegro99-251/sweater...
1,1161236,ALEGRO99.,Jaket Sweater ALEGRO99 JPG Hodie Polos Ori Pri...,Rp230.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/alegro99-251/jaket-s...
2,1161237,ALEGRO99.,Alegro99 Keep Quit Celana Pendek Jumbo Cargo P...,Rp45.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/alegro99-251/alegro9...
3,1161238,ALEGRO99.,Alegro99 Spider Celana Pendek Jumbo Cargo Pri...,Rp45.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/alegro99-251/alegro9...
4,1161239,ALEGRO99.,Alegro99 Stay Smoke Celana Pendek Jumbo Cargo...,Rp45.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/alegro99-251/alegro9...
...,...,...,...,...,...,...,...
16419,1177654,Apotek Golden Sehat by GoApotik,FIESTA KONDOM STRAWBERRY BOX 3 PCS,Rp16.755,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-golden-sehat-...
16420,1177655,Apotek Golden Sehat by GoApotik,DUMIN SYRUP ISI 60 ML BOTOL,Rp42.081,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-golden-sehat-...
16421,1177656,Apotek Golden Sehat by GoApotik,DAKTARIN DIAPERS SALEP 10 GRAM,Rp71.694,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-golden-sehat-...
16422,1177657,Apotek Golden Sehat by GoApotik,DUMIN 250 MG RECTAL TUBE 4 ML,Rp41.433,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-golden-sehat-...


In [ ]:
double_shop = set(df1["Shop_Name"].unique()).intersection(set(df2["Shop_Name"].unique()))
list_double_shop = list(double_shop)
print(f"Total Shop in two DataFrame: {len(list_double_shop)}")
print(list_double_shop)

Total Shop in two DataFrame: 14
['Amarta Herbal Store', 'Apotek azka langkat by Goa', 'oraimo Authentic Store', 'VIVA APOTEK', 'Apotek Azzahra 24 by GoApotik', 'Apotek Halomedika Pekayon', 'Apotek Halomedika Kranggan', 'apotekharapansehat', 'Alcavella', 'Apotek Alkafi Farma Mojokerto by GoApotik', 'Apotek Kreator Kesehatan Cipondoh', 'Apotek Golden Sehat by GoApotik', 'Apotek Helios Medika Saharjo', 'Apotek Athar Farma Palembang by GoApotik']


In [ ]:
df1_cleaned = df1[~df1["Shop_Name"].isin(list_double_shop)]
df1_cleaned

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,1,Awkward Brand,Kaos Polos Anak Katun Combed 30s Hijau Olive,Rp74.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
1,2,Awkward Brand,Kaos Polos Anak Katun Combed 30s Biru Navy,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
2,3,Awkward Brand,Kaos Polos Anak Katun Combed 30s Kuning Mustard,Rp74.000,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
3,4,Awkward Brand,Kaos Polos Anak Katun Combed 30s Merah Maroon,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
4,5,Awkward Brand,Kaos Polos Anak Katun Combed 30s Abu-abu Misty,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
...,...,...,...,...,...,...,...
1161229,1161230,PKB Bandung,VIVA ASTRINGENT CUCUMBER 200 ML Toner Wajah,Rp12.927,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/pkb-bandung/viva-ast...
1161230,1161231,PKB Bandung,Viva Anti Winkel Cream 22 gr,Rp13.857,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/pkb-bandung/viva-ant...
1161231,1161232,PKB Bandung,VIVA AIR MAWAR 200 ML | ROSE WATER Bunga Peny...,Rp12.071,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/pkb-bandung/viva-air...
1161232,1161233,PKB Bandung,Viva Face Tonic Green Tea 100 ml Berjerawat P...,Rp9.114,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/pkb-bandung/viva-fac...


Identifying and removing duplicate shop names during the Phase 3 scraping process.

In [ ]:
df3

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,1177659,By.A Herbal,Sendifit Asli Obat Asam Urat Dan Rematik (100%...,Rp137.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/sendifit-...
1,1177660,By.A Herbal,Cara Mengobati Asam Lambung Dengan Kapsul Seha...,Rp125.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/cara-meng...
2,1177661,By.A Herbal,hanya packing herbal keloreena,Rp999.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/hanya-pac...
3,1177662,By.A Herbal,"Obat Sesak Di Dada , Nafas Berat, Nyeri Ulu Ha...",Rp125.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/obat-sesa...
4,1177663,By.A Herbal,Kopi Herbal Radimax Isi 10 Sachet Sudah BPOM &...,Rp150.000,4.3,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/kopi-herb...
...,...,...,...,...,...,...,...
1016199,2193858,Fimory Mall,Beli dengan voucher Fimory Gastro Original Pak...,Rp285.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/beli-den...
1016200,2193859,Fimory Mall,Fimory Gastro Original Paket 5 Box - Minuman C...,Rp475.000,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...
1016201,2193860,Fimory Mall,Fimory Gastro Best Seller Paket 2 Box - Cereal...,Rp194.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...
1016202,2193861,Fimory Mall,Fimory Gastro Original Paket 6 Box - Minuman S...,Rp570.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...


In [ ]:
df4

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,2193863,Mizzu Bottle Official,Mizzu Bottle - Brew Botol Minum Stainless Cera...,Rp212.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mizzu-bottle-officia...
1,2193864,Mizzu Bottle Official,Mizzu Bottle - Flow Botol Minum Stainless Tumb...,Rp210.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mizzu-bottle-officia...
2,2193865,Mizzu Bottle Official,Mizzu Bottle - Barrel Botol Minum Stainless Tu...,Rp267.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mizzu-bottle-officia...
3,2193866,Mizzu Bottle Official,Mizzu Bottle - Urban Tumbler Botol Minum Stain...,Rp145.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mizzu-bottle-officia...
4,2193867,Mizzu Bottle Official,Mizzu Grip Tumbler Besar Botol Minum Stainless...,Rp159.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mizzu-bottle-officia...
...,...,...,...,...,...,...,...
23358,2217221,Happy Family Storee,(HF) LOKAL Keset Lantai Cushion Printing Anti ...,Rp25.200,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/happyfamilyofficial/...
23359,2217222,Happy Family Storee,(HF) LOKAL Keset Lantai Glossy Nyerap Air Moti...,Rp21.100,4.6,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/happyfamilyofficial/...
23360,2217223,Happy Family Storee,(HF) LOKAL Keset Lantai MOTIF Modern UK 40x60...,Rp19.600,4.8,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/happyfamilyofficial/...
23361,2217224,Happy Family Storee,(HF) SNI Playmat Lipat Anak XPE Matras Karpet ...,Rp106.000,4.8,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/happyfamilyofficial/...


In [ ]:
double_shop1 = set(df3["Shop_Name"].unique()).intersection(set(df4["Shop_Name"].unique()))
list_double_shop1 = list(double_shop1)
print(f"Total Shop in two DataFrame: {len(list_double_shop1)}")
print(list_double_shop1)

Total Shop in two DataFrame: 14
['Berkah Saluyu', 'FixPrint Indonesia', 'Bintang Official Store', 'COLORKEY INDONESIA', 'Darrie Store', 'Mizzu Bottle Official', 'Century Healthcare Express', 'Happy Family Storee', 'Hyfresh Bojongsari', 'Apotek Aiman by GoApotik', 'DUST SHOP', 'Bakpia Tugu Jogja', 'Apotek Yenni by GoApotik', 'Hypermart Balekota']


In [ ]:
df3_cleaned = df3[~df3["Shop_Name"].isin(list_double_shop1)]
df3_cleaned

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,1177659,By.A Herbal,Sendifit Asli Obat Asam Urat Dan Rematik (100%...,Rp137.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/sendifit-...
1,1177660,By.A Herbal,Cara Mengobati Asam Lambung Dengan Kapsul Seha...,Rp125.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/cara-meng...
2,1177661,By.A Herbal,hanya packing herbal keloreena,Rp999.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/hanya-pac...
3,1177662,By.A Herbal,"Obat Sesak Di Dada , Nafas Berat, Nyeri Ulu Ha...",Rp125.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/obat-sesa...
4,1177663,By.A Herbal,Kopi Herbal Radimax Isi 10 Sachet Sudah BPOM &...,Rp150.000,4.3,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bya-herbal/kopi-herb...
...,...,...,...,...,...,...,...
1016199,2193858,Fimory Mall,Beli dengan voucher Fimory Gastro Original Pak...,Rp285.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/beli-den...
1016200,2193859,Fimory Mall,Fimory Gastro Original Paket 5 Box - Minuman C...,Rp475.000,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...
1016201,2193860,Fimory Mall,Fimory Gastro Best Seller Paket 2 Box - Cereal...,Rp194.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...
1016202,2193861,Fimory Mall,Fimory Gastro Original Paket 6 Box - Minuman S...,Rp570.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fimory-mall/fimory-g...


In [ ]:
df5

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,2217226,Weimar Factory,Njonjamar - Kue Kacang Wijen Kucang Jadul Kue ...,Rp 30.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://tokopedia.com/weimarofficial/njonjamar...
1,2217227,Weimar Factory,Binbon Stick Gemoy - Stik Gemoyy Coklat @250 G...,Rp 25.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://tokopedia.com/weimarofficial/binbon-st...
2,2217228,Weimar Factory,Binbon Stick Gemoy - Stik Gemoy KEJU @250 Gr R...,Rp 25.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://tokopedia.com/weimarofficial/binbon-st...
3,2217229,Weimar Factory,Weimar x Luvme Choco Jelly Fruit Jam - Coklat ...,Rp 50.000,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://tokopedia.com/weimarofficial/weimar-x-...
4,2217230,Weimar Factory,Binbon Jet Kentang - Rasa Original @200 Gr Sna...,Rp 15.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://tokopedia.com/weimarofficial/binbon-je...
...,...,...,...,...,...,...,...
775745,2992971,LJS Phone,Apple iPhone 17 256 GB 512GB Garansi Resmi In...,Rp17.399.000,5.0,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ljsofficial/apple-ip...
775746,2992972,LJS Phone,tempered glass redmi note 8,Rp35.000,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ljsofficial/tempered...
775747,2992973,LJS Phone,Tempered Glass Xiaomi Redmi 6A Full 5D,Rp50.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ljsofficial/tempered...
775748,2992974,LJS Phone,XTRA BUBBLE WRAP,Rp5.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ljsofficial/xtra-bub...


Identifying and removing duplicate shop names during the Phase 6 scraping process.

In [ ]:
df6

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,2992976,Lenovo Legion Official,Lenovo IdeaCentre Tower 08IRH9 I5 13420H 8GB 5...,Rp11.949.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/lenovolegion/lenovo-...
1,2992977,Lenovo Legion Official,LENOVO MONITOR 27 INCH L27Q-11 IPS QHD 2K 1440...,Rp3.879.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/lenovolegion/lenovo-...
2,2992978,Lenovo Legion Official,LENOVO PC IDEACENTRE MINI INTEL CORE 5 210H 8G...,Rp11.949.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/lenovolegion/lenovo-...
3,2992979,Lenovo Legion Official,LENOVO LOQ 15 RYZEN 7 7735HS 16GB 512GB RTX405...,Rp17.999.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/lenovolegion/lenovo-...
4,2992980,Lenovo Legion Official,LENOVO LEGION GAMING MONITOR 24 INCH LEGION 24...,Rp1.869.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/lenovolegion/lenovo-...
...,...,...,...,...,...,...,...
636149,3629125,Zoya,Zoya HARDBOX Scarf - Box Hampers Scarf [BOX SAJA],Rp25.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/zoya-hardbox-sc...
636150,3629126,Zoya,Kerudung Hijab Instan Bergo - Zoya Queena Berg...,Rp79.200,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/kerudung-hijab-...
636151,3629127,Zoya,Zoya - Kamaniya Scarf - Kerudung Hijab Segiemp...,Rp44.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/zoya-kamaniya-s...
636152,3629128,Zoya,Zoya Bergo Hijab Instant Marsha Glittering Cas...,Rp107.100,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/zoya-bergo-mars...


In [ ]:
df7

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,3629130,Laviola Shoes,Laviola 5299 LSH - Sepatu Flat Wanita Beige da...,Rp215.910,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/laviolashoes/laviola...
1,3629131,Laviola Shoes,Laviola 5298 LSH - Sepatu Flat Wanita BLACK da...,Rp215.910,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/laviolashoes/laviola...
2,3629132,Laviola Shoes,Laviola 5293 LSH - Sepatu Flat Wanita Black Be...,Rp215.910,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/laviolashoes/laviola...
3,3629133,Laviola Shoes,Laviola 5292 LSH - Sepatu Flat Wanita Black Ap...,Rp215.910,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/laviolashoes/laviola...
4,3629134,Laviola Shoes,Laviola 5239 LSF - Sepatu Flat Wanita Black da...,Rp215.910,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/laviolashoes/laviola...
...,...,...,...,...,...,...,...
10198,3639328,Apotek Ubay Farma by GoApotik,COMBIVENT UDV CAIRAN INHALASI 2.5 ML PACK 10 VIAL,Rp66.526,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
10199,3639329,Apotek Ubay Farma by GoApotik,COMBIVENT UDV CAIRAN INHALASI 2.5 ML BOX 20 VIAL,Rp114.908,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
10200,3639330,Apotek Ubay Farma by GoApotik,FLUIMUCIL 100 MG/ML CAIRAN INHALASI 3 ML AMPUL,Rp23.138,5.0,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
10201,3639331,Apotek Ubay Farma by GoApotik,FLUIMUCIL 100 MG/ML CAIRAN INHALASI 3 ML BOX 5...,Rp109.595,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...


In [ ]:
double_shop2 = set(df6["Shop_Name"].unique()).intersection(set(df7["Shop_Name"].unique()))
list_double_shop2 = list(double_shop2)
print(f"Total Shop in two DataFrame: {len(list_double_shop2)}")
print(list_double_shop2)

Total Shop in two DataFrame: 8
['SIM XPRESS', 'UniGet', 'tokostd_NEW', 'Tennesy', 'WELLEN PRINT OFFICIAL', 'Apotek Ubay Farma by GoApotik', 'Laviola Shoes', 'Menta Toys']


In [ ]:
df6_cleaned = df6[~df6["Shop_Name"].isin(list_double_shop2)]
df6_cleaned

,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,2992976,Lenovo Legion Official,Lenovo IdeaCentre Tower 08IRH9 I5 13420H 8GB 5...,Rp11.949.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/lenovolegion/lenovo-...
1,2992977,Lenovo Legion Official,LENOVO MONITOR 27 INCH L27Q-11 IPS QHD 2K 1440...,Rp3.879.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/lenovolegion/lenovo-...
2,2992978,Lenovo Legion Official,LENOVO PC IDEACENTRE MINI INTEL CORE 5 210H 8G...,Rp11.949.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/lenovolegion/lenovo-...
3,2992979,Lenovo Legion Official,LENOVO LOQ 15 RYZEN 7 7735HS 16GB 512GB RTX405...,Rp17.999.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/lenovolegion/lenovo-...
4,2992980,Lenovo Legion Official,LENOVO LEGION GAMING MONITOR 24 INCH LEGION 24...,Rp1.869.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/lenovolegion/lenovo-...
...,...,...,...,...,...,...,...
636149,3629125,Zoya,Zoya HARDBOX Scarf - Box Hampers Scarf [BOX SAJA],Rp25.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/zoya-hardbox-sc...
636150,3629126,Zoya,Kerudung Hijab Instan Bergo - Zoya Queena Berg...,Rp79.200,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/kerudung-hijab-...
636151,3629127,Zoya,Zoya - Kamaniya Scarf - Kerudung Hijab Segiemp...,Rp44.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/zoya-kamaniya-s...
636152,3629128,Zoya,Zoya Bergo Hijab Instant Marsha Glittering Cas...,Rp107.100,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/zoya/zoya-bergo-mars...


Merging All Datasets

In [ ]:
list_df = [df1_cleaned, df2, df3_cleaned, df4, df5, df6_cleaned, df7]
df_final = pd.concat(list_df, axis=0)
df_final = df_final.reset_index(drop=True)

In [ ]:
df_final.shape

(3614752, 7)

In [ ]:
df_final['ID_Product'] = range(1, len(df_final) + 1)
col_to_move = df_final.pop('ID_Product')
df_final.insert(0, 'ID_Product', col_to_move)
df_final

,ID_Product,Id,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,1,1,Awkward Brand,Kaos Polos Anak Katun Combed 30s Hijau Olive,Rp74.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
1,2,2,Awkward Brand,Kaos Polos Anak Katun Combed 30s Biru Navy,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
2,3,3,Awkward Brand,Kaos Polos Anak Katun Combed 30s Kuning Mustard,Rp74.000,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
3,4,4,Awkward Brand,Kaos Polos Anak Katun Combed 30s Merah Maroon,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
4,5,5,Awkward Brand,Kaos Polos Anak Katun Combed 30s Abu-abu Misty,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
...,...,...,...,...,...,...,...,...
3614747,3614748,3639328,Apotek Ubay Farma by GoApotik,COMBIVENT UDV CAIRAN INHALASI 2.5 ML PACK 10 VIAL,Rp66.526,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
3614748,3614749,3639329,Apotek Ubay Farma by GoApotik,COMBIVENT UDV CAIRAN INHALASI 2.5 ML BOX 20 VIAL,Rp114.908,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
3614749,3614750,3639330,Apotek Ubay Farma by GoApotik,FLUIMUCIL 100 MG/ML CAIRAN INHALASI 3 ML AMPUL,Rp23.138,5.0,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
3614750,3614751,3639331,Apotek Ubay Farma by GoApotik,FLUIMUCIL 100 MG/ML CAIRAN INHALASI 3 ML BOX 5...,Rp109.595,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...


In [ ]:
df_final.drop("Id",axis=1, inplace=True)
df_final

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product
0,1,Awkward Brand,Kaos Polos Anak Katun Combed 30s Hijau Olive,Rp74.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
1,2,Awkward Brand,Kaos Polos Anak Katun Combed 30s Biru Navy,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
2,3,Awkward Brand,Kaos Polos Anak Katun Combed 30s Kuning Mustard,Rp74.000,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
3,4,Awkward Brand,Kaos Polos Anak Katun Combed 30s Merah Maroon,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
4,5,Awkward Brand,Kaos Polos Anak Katun Combed 30s Abu-abu Misty,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...
...,...,...,...,...,...,...,...
3614747,3614748,Apotek Ubay Farma by GoApotik,COMBIVENT UDV CAIRAN INHALASI 2.5 ML PACK 10 VIAL,Rp66.526,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
3614748,3614749,Apotek Ubay Farma by GoApotik,COMBIVENT UDV CAIRAN INHALASI 2.5 ML BOX 20 VIAL,Rp114.908,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
3614749,3614750,Apotek Ubay Farma by GoApotik,FLUIMUCIL 100 MG/ML CAIRAN INHALASI 3 ML AMPUL,Rp23.138,5.0,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...
3614750,3614751,Apotek Ubay Farma by GoApotik,FLUIMUCIL 100 MG/ML CAIRAN INHALASI 3 ML BOX 5...,Rp109.595,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...


In [ ]:
df_final.to_csv(f'/content/drive/MyDrive/df_final.csv', index=False)

# download images

In [ ]:
import pandas as pd
from google.colab import drive
import requests
import pandas as pd
import os, re, time, threading, math
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from google.colab import files
import json

drive.mount('/content/drive')
df_final = pd.read_csv("/content/drive/MyDrive/df_final.csv")
df_sid = pd.read_csv("/content/drive/MyDrive/Scrapping_list_shop_with_SID.csv")

Mounted at /content/drive


In [ ]:
df_final = df_final.merge(df_sid[['Nama Toko', 'SID']], left_on='Shop_Name', right_on='Nama Toko', how='left')
df_final.drop(columns=['Nama Toko'], inplace=True)

In [ ]:
path_folder = '/content/images_product'
batch_size  = 500
thread_count = 20
os.makedirs(path_folder, exist_ok=True)
write_lock = threading.Lock()
error_list = []

headers = {
    'sec-ch-ua-platform': '"Windows"',
    'x-version': '17e0d90',
    'Referer': 'https://www.tokopedia.com/',
    'sec-ch-ua': '"Google Chrome";v="147", "Not.A/Brand";v="8", "Chromium";v="147"',
    'x-price-center': 'true',
    'sec-ch-ua-mobile': '?0',
    'bd-device-id': '7628337271406577160',
    'x-source': 'tokopedia-lite',
    'x-device': 'default_v3',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36',
    'accept': '*/*',
    'content-type': 'application/json',
    'x-tkpd-lite-service': 'zeus'
}

def mark_error(product_id):
    with write_lock:
        error_list.append(product_id)

def clean_url(url):
    return url.split('?')[0].rstrip('/')

def get_keyword(product_name):
    words = str(product_name).split()
    return ' '.join(words[:3])

def get_image_url(sid, product_name, target_url, retries=3, session=None):
    if session is None:
        session = requests.Session()
    keyword      = get_keyword(product_name)
    target_clean = clean_url(target_url)
    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": str(sid), "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]
    for _ in range(retries):
        try:
            res = session.post(
                'https://gql.tokopedia.com/graphql/ShopProducts',
                headers=headers, json=payload, timeout=15
            )
            if res.status_code != 200:
                continue
            products = res.json()[0]['data']['GetShopProduct']['data']
            if not products:
                return None
            for p in products:
                if clean_url(p.get('product_url', '')) == target_clean:
                    return p.get('primary_image', {}).get('original')
            return products[0].get('primary_image', {}).get('original')
        except Exception:
            pass
    return None

def process_row(row):
    session      = requests.Session()
    product_id   = str(row["ID_Product"])
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    url          = row["URL_Product"]
    filepath     = os.path.join(path_folder, f"{product_id}.jpg")
    if os.path.exists(filepath):
        return "skip"
    img_url = get_image_url(sid, product_name, url, session=session)
    if not img_url:
        mark_error(product_id)
        return "error"
    for _ in range(3):
        try:
            img_res = session.get(
                img_url,
                headers={'User-Agent': headers['User-Agent']},
                timeout=10
            )
            if img_res.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(img_res.content)
                return "success"
        except Exception:
            pass
    mark_error(product_id)
    return "error"

## phase 1

In [ ]:
df_final1 = df_final.loc[:150000]

In [ ]:
df_final1.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
0,1,Awkward Brand,Kaos Polos Anak Katun Combed 30s Hijau Olive,Rp74.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...,14656547
1,2,Awkward Brand,Kaos Polos Anak Katun Combed 30s Biru Navy,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...,14656547
2,3,Awkward Brand,Kaos Polos Anak Katun Combed 30s Kuning Mustard,Rp74.000,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...,14656547
3,4,Awkward Brand,Kaos Polos Anak Katun Combed 30s Merah Maroon,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...,14656547
4,5,Awkward Brand,Kaos Polos Anak Katun Combed 30s Abu-abu Misty,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...,14656547


In [ ]:
done_ids = set(f.replace(".jpg", "")for f in os.listdir(path_folder)if f.endswith(".jpg"))
df_todo = df_final1[~df_final1["ID_Product"].astype(str).isin(done_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 150001/150001 [10:36:19<00:00,  3.93product/s, batch=301/301, Success: =137436, Fail: =12565]


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final1[~df_final1['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
42,43,Apotek Sengeti Farma Sekernan,ALLOPURINOL 100MG 1 STRIP 10 TABLET (Gen HJ),Rp4.976,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
43,44,Apotek Sengeti Farma Sekernan,INTERHISTIN 50MG 1 STRIP ISI 10 TABLET,Rp16.813,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
44,45,Apotek Sengeti Farma Sekernan,IMODIUM 2MG 1 STRIP 10 TABLET,Rp143.783,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
45,46,Apotek Sengeti Farma Sekernan,FARMOTEN 25MG 1 STRIP 10 TABLET,Rp5.499,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
51,52,Apotek Sengeti Farma Sekernan,"HARNAL OCAS 0,4MG 1 BLISTER 10 TABLET",Rp161.068,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
...,...,...,...,...,...,...,...,...
149894,149895,Apotek SBS Farma Pandu Raya,Otozambon Ear Drops Botol,Rp77.779,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
149951,149952,Apotek SBS Farma Pandu Raya,Regumen 5 mg 10 Tablet Strip,Rp57.204,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
149965,149966,Apotek SBS Farma Pandu Raya,Avocel 5 mg 10 Tablet - Obat Alergi - Halodoc,Rp64.544,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
149977,149978,Apotek SBS Farma Pandu Raya,Concor 1.25 mg 10 Tablet - Hipertensi - Halodoc,Rp49.058,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991


In [ ]:
done_ids = set(f.replace(".jpg", "")for f in os.listdir(path_folder)if f.endswith(".jpg"))
df_todo = error_data[~error_data["ID_Product"].astype(str).isin(done_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 12565/12565 [29:19<00:00,  7.14product/s, batch=26/26, Success: =1672, Fail: =10893]


In [ ]:
drive.mount('/content/drive')
zip_path = "/content/drive/MyDrive/Data_Images1/images_product.zip"
destination_folder = "/content/images_product"
os.makedirs(destination_folder, exist_ok=True)
!unzip -q "{zip_path}" -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
existing_images1 = {f.split('.')[0] for f in os.listdir('images_product')}
len(existing_images1)

139108

In [ ]:
error_data1 = df_final1[~df_final1['ID_Product'].astype(str).isin(existing_images1)]
error_data1

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
42,43,Apotek Sengeti Farma Sekernan,ALLOPURINOL 100MG 1 STRIP 10 TABLET (Gen HJ),Rp4.976,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
43,44,Apotek Sengeti Farma Sekernan,INTERHISTIN 50MG 1 STRIP ISI 10 TABLET,Rp16.813,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
44,45,Apotek Sengeti Farma Sekernan,IMODIUM 2MG 1 STRIP 10 TABLET,Rp143.783,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
45,46,Apotek Sengeti Farma Sekernan,FARMOTEN 25MG 1 STRIP 10 TABLET,Rp5.499,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
51,52,Apotek Sengeti Farma Sekernan,"HARNAL OCAS 0,4MG 1 BLISTER 10 TABLET",Rp161.068,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
...,...,...,...,...,...,...,...,...
149785,149786,Apotek SBS Farma Pandu Raya,Siladex Cough & Cold 100 ml,Rp21.704,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
149951,149952,Apotek SBS Farma Pandu Raya,Regumen 5 mg 10 Tablet Strip,Rp57.204,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
149965,149966,Apotek SBS Farma Pandu Raya,Avocel 5 mg 10 Tablet - Obat Alergi - Halodoc,Rp64.544,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
149977,149978,Apotek SBS Farma Pandu Raya,Concor 1.25 mg 10 Tablet - Hipertensi - Halodoc,Rp49.058,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991


In [ ]:
available_ids   = []
unavailable_ids = []

def check_image_available(row):
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    keyword      = ' '.join(str(product_name).split()[:3])
    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]
    try:
        res      = requests.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        return "ada" if products else "kosong"
    except:
        return "kosong"
rows = error_data1.to_dict("records")
for row in tqdm(rows, unit="produk"):
    product_id = str(row["ID_Product"])
    status     = check_image_available(row)
    if status == "ada":
        available_ids.append(product_id)
    else:
        unavailable_ids.append(product_id)
print(f"Total Available Images : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 10893/10893 [1:40:50<00:00,  1.80produk/s]

Total Available Images : 32 (1066.7%)
Total Unavailable Images : 10,861 (362033.3%)


In [ ]:
data_unavailable = error_data1[error_data1['ID_Product'].astype(str).isin(unavailable_ids)]
available_ids1   = []
unavailable_ids1 = []

def check_image_available(row):
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    keyword      = ' '.join(str(product_name).split()[:3])
    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]
    try:
        res      = requests.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        return "ada" if products else "kosong"
    except:
        return "kosong"
rows = data_unavailable.to_dict("records")

for row in tqdm(rows, unit="produk"):
    product_id1 = str(row["ID_Product"])
    status1     = check_image_available(row)
    if status1 == "ada":
        available_ids1.append(product_id1)
    else:
        unavailable_ids1.append(product_id1)
print(f"Total Available Images : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 10861/10861 [1:38:15<00:00,  1.84produk/s]

Total Available Images : 32 (100.0%)
Total Unavailable Images : 10,861 (33940.6%)


In [ ]:
df_final.loc[16566,'URL_Product']

'https://www.tokopedia.com/apotek-rave-pharma-by-goapotik/flagystatin-strip-isi-5-ovula?extParam=src%3Dshop%26whid%3D11511533'

In [ ]:
available_ids

['4544',
 '5178',
 '8057',
 '32199',
 '35515',
 '35711',
 '36171',
 '36462',
 '38009',
 '38040',
 '38239',
 '38427',
 '38428',
 '38528',
 '38628',
 '38634',
 '38856',
 '38879',
 '38904',
 '39242',
 '75743',
 '88708',
 '88725',
 '88726',
 '103696',
 '146192',
 '148729',
 '148805',
 '148860',
 '149172',
 '149479',
 '149681']

In [ ]:
available_ids

['32199', '75743', '103696']

In [ ]:
error_data2 = df_final1[df_final1['ID_Product'].astype(str).isin(available_ids)]
error_data2

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
4543,4544,AJM Electronic Store,INKAC【COD】llano V12 Cooler Pad RGB Laptop Cool...,Rp1.560.000,4.8,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ajm-electronic-store...,7495922725109729982
5177,5178,Ugreen Computer Acc,UGREEN Barel RJ45 Sambungan Kabel Lan Cat5 Cat...,Rp44.500,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ugreencomputeracc/ug...,6522719
8056,8057,ASTEROS OFFICIAL,MCDODO TWS B04 Series Earbuds With Digital Dis...,Rp215.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/asterosofficial/mcdo...,13346863
32198,32199,Apotek Manjur Sehat Farma By GoApotik,ACCU-CHEK ACTIVE STRIP GULA DARAH 50 PCS,Rp384.685,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-manjur-sehat-...,16854476
35514,35515,Apotekmandjurofficial,Enervon-C Active 4 Tablet - Suplemen Lengkap u...,Rp7.700,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/mandjur/enervon-c-ac...,824445
35710,35711,Apotekmandjurofficial,Triplixam 10 mg/2.5 mg/10 mg Botol 30 Tablet,Rp813.350,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mandjur/triplixam-10...,824445
36170,36171,Apotekmandjurofficial,Lapisiv-T 10 Tablet - Obat Batuk Pilek Alergi,Rp35.400,4.8,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mandjur/lapisiv-t-10...,824445
36461,36462,Apotekmandjurofficial,Astaxanthin Natural Mpl 4 mg 6 Kapsul - Suplem...,Rp26.400,4.9,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/mandjur/astaxanthin-...,824445
38008,38009,Apotekmandjurofficial,Enervon Gold 5 Kapsul - Multivitamin Khusus La...,Rp22.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mandjur/enervon-gold...,824445
38039,38040,Apotekmandjurofficial,Prive Uricran 10 Kapsul - Suplemen Kesehatan S...,Rp78.200,4.9,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/mandjur/prive-uricra...,824445


In [ ]:
done_ids = set(f.replace(".jpg", "")for f in os.listdir(path_folder)if f.endswith(".jpg"))
df_todo = error_data2.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 32/32 [00:13<00:00,  2.45product/s, batch=1/1, Success: =28, Fail: =2]


In [ ]:
error_list

['103696', '103696', '103696', '4544', '103696']

In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir(folder_path) if f.endswith('.jpg')}
error_data2['is_exists'] = error_data2['ID_Product'].astype(str).isin(existing_files)
missing_data = error_data2[~error_data2['is_exists']]
still_missing_list = missing_data['ID_Product'].tolist()
print(f"Total Images: {len(error_data2)}")
print(f"Downloaded: {len(existing_files)}")
print(f"Still Missing: {len(still_missing_list)}")

Total Images: 32
Downloaded: 139138
Still Missing: 2


/tmp/ipykernel_2007/3848048087.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  error_data2['is_exists'] = error_data2['ID_Product'].astype(str).isin(existing_files)


In [ ]:
missing_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID,is_exists
4543,4544,AJM Electronic Store,INKAC【COD】llano V12 Cooler Pad RGB Laptop Cool...,Rp1.560.000,4.8,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ajm-electronic-store...,7495922725109729982,False
103695,103696,ALFAUZI COLLECTION,PESANAN PUNYA IBU NINA ACEH,Rp200.000,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/alfauzicolection/pes...,1324256,False


In [ ]:
missing_id_product = df_final1[~df_final1["ID_Product"].astype(str).isin(existing_files)].copy()
missing_id_product

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
42,43,Apotek Sengeti Farma Sekernan,ALLOPURINOL 100MG 1 STRIP 10 TABLET (Gen HJ),Rp4.976,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
43,44,Apotek Sengeti Farma Sekernan,INTERHISTIN 50MG 1 STRIP ISI 10 TABLET,Rp16.813,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
44,45,Apotek Sengeti Farma Sekernan,IMODIUM 2MG 1 STRIP 10 TABLET,Rp143.783,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
45,46,Apotek Sengeti Farma Sekernan,FARMOTEN 25MG 1 STRIP 10 TABLET,Rp5.499,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
51,52,Apotek Sengeti Farma Sekernan,"HARNAL OCAS 0,4MG 1 BLISTER 10 TABLET",Rp161.068,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
...,...,...,...,...,...,...,...,...
149785,149786,Apotek SBS Farma Pandu Raya,Siladex Cough & Cold 100 ml,Rp21.704,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
149951,149952,Apotek SBS Farma Pandu Raya,Regumen 5 mg 10 Tablet Strip,Rp57.204,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
149965,149966,Apotek SBS Farma Pandu Raya,Avocel 5 mg 10 Tablet - Obat Alergi - Halodoc,Rp64.544,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
149977,149978,Apotek SBS Farma Pandu Raya,Concor 1.25 mg 10 Tablet - Hipertensi - Halodoc,Rp49.058,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase1.json', 'w') as f:
    json.dump(missing_ids, f)

files.download('totally_no_iamges_during_phase1.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase1.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 2

In [ ]:
df_final2 = df_final.loc[150000:300000]
df_final2

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
150000,150001,Apotek SBS Farma Pandu Raya,Desolex 0.05% Cream 10 g Tube,Rp28.571,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150001,150002,Apotek SBS Farma Pandu Raya,Cendo Posop Minidose 0.6 ml,Rp61.643,4.9,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150002,150003,Apotek SBS Farma Pandu Raya,Hydrocortisone Cream 1 % 5 g,Rp8.172,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150003,150004,Apotek SBS Farma Pandu Raya,Interbi 1% Cream 5 g Tube,Rp37.847,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150004,150005,Apotek SBS Farma Pandu Raya,Ezetrol 10 mg 10 Tablet Strip,Rp263.876,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
...,...,...,...,...,...,...,...,...
299996,299997,Apotek Sultan Palembang by GoApotik,HOLISTICARE SUPER ESTER-C STRIP 4 TABLET,Rp9.134,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
299997,299998,Apotek Sultan Palembang by GoApotik,CDR FORTOS EFFERVESCENT 10 TABLET,Rp63.931,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
299998,299999,Apotek Sultan Palembang by GoApotik,CALADINE LOTION 60 ML BOTOL,Rp22.110,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
299999,300000,Apotek Sultan Palembang by GoApotik,CDR RASA JERUK MANDARIN TUBE 15 TABLET,Rp87.310,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109


In [ ]:
done_ids = set(f.replace(".jpg", "")for f in os.listdir(path_folder)if f.endswith(".jpg"))
df_todo = df_final2[~df_final2["ID_Product"].astype(str).isin(done_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 150001/150001 [9:06:14<00:00,  4.58product/s, batch=301/301, Success: =141042, Fail: =8959]


In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

In [ ]:
drive.mount('/content/drive')
zip_path = "/content/drive/MyDrive/Data_Images1/images_product.zip"
destination_folder = "/content/images_product"
os.makedirs(destination_folder, exist_ok=True)
!unzip -q "{zip_path}" -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final2[~df_final2['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
150009,150010,Apotek SBS Farma Pandu Raya,Methycobal 500 mcg 10 Kapsul,Rp51.357,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150011,150012,Apotek SBS Farma Pandu Raya,Thiamycin 500 mg 10 Kapsul,Rp82.314,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150035,150036,Apotek SBS Farma Pandu Raya,Flunarizine 10 mg 10 Tablet - Obat Pereda Dema...,Rp26.997,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150052,150053,Apotek SBS Farma Pandu Raya,Cendo Catarlent Eye Drops 15 ml,Rp35.959,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150096,150097,Apotek SBS Farma Pandu Raya,Gliquidone 30 mg 10 Tablet,Rp18.416,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
...,...,...,...,...,...,...,...,...
299754,299755,Apotek Laras Farma Payaman Magelang by GoApotik,CESSA KIDS BOFIT 8 ML BOTOL,Rp41.404,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-laras-farma-p...,18033543
299882,299883,Apotek Laras Farma Payaman Magelang by GoApotik,OB HERBAL JUNIOR SYRUP ISI 30 ML BOTOL,Rp10.961,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-laras-farma-p...,18033543
299957,299958,Apotek Laras Farma Payaman Magelang by GoApotik,IPI VITAMIN C 50 MG 50 TABLET,Rp9.134,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-laras-farma-p...,18033543
299960,299961,Apotek Sultan Palembang by GoApotik,HOTIN CREAM 120 GRAM TUBE,Rp43.596,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109


In [ ]:
available_ids   = []
unavailable_ids = []

def check_image_available(row):
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]
    try:
        res      = requests.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        return "ada" if products else "kosong"
    except:
        return "kosong"
rows = error_data.to_dict("records")

for row in tqdm(rows, unit="produk"):
    product_id = str(row["ID_Product"])
    status     = check_image_available(row)
    if status == "ada":
        available_ids.append(product_id)
    else:
        unavailable_ids.append(product_id)

100%|██████████| 8959/8959 [1:31:07<00:00,  1.64produk/s]


NameError: name 'total' is not defined

In [ ]:
total   = len(rows)
print(f"Total Available Images : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

Total Available Images : 2,400 (26.8%)
Total Unavailable Images : 6,559 (73.2%)


In [ ]:
available_ids[:5]

['150036', '150097', '150646', '150675', '150754']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
150035,150036,Apotek SBS Farma Pandu Raya,Flunarizine 10 mg 10 Tablet - Obat Pereda Dema...,Rp26.997,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150096,150097,Apotek SBS Farma Pandu Raya,Gliquidone 30 mg 10 Tablet,Rp18.416,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150645,150646,Apotek SBS Farma Pandu Raya,Tantum Verde Oral Rinse 120 ml,Rp42.622,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150674,150675,ATK & Co,Dahlia Blue Clean Pembersih Kloset BC002 Isi 3...,Rp22.145,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/atknco/dahlia-blue-c...,8594694
150753,150754,ATK & Co,Digital Finger Counter Joyko FGCT-2022 - Mini ...,Rp8.965,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/atknco/digital-finge...,8594694
...,...,...,...,...,...,...,...,...
299517,299518,Apotek Rafi Palembang by GoApotik,FRESHCARE SMASH AROMATHERAPY 8 ML BOTOL,Rp17.050,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-rafi-palemban...,17637108
299725,299726,Apotek Rafi Palembang by GoApotik,NEUROBION STRIP 10 TABLET,Rp35.069,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-rafi-palemban...,17637108
299726,299727,Apotek Rafi Palembang by GoApotik,FOLAMIL GOLD BOTOL 30 KAPSUL,Rp227.291,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-rafi-palemban...,17637108
299882,299883,Apotek Laras Farma Payaman Magelang by GoApotik,OB HERBAL JUNIOR SYRUP ISI 30 ML BOTOL,Rp10.961,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-laras-farma-p...,18033543


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 2400/2400 [10:35<00:00,  3.78product/s, batch=5/5, Success: =2378, Fail: =22]


In [ ]:
error_listv2 = error_list
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_listv2)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 22/22 [00:07<00:00,  2.89product/s, batch=1/1, Success: =18, Fail: =4]


In [ ]:
error_data.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
150009,150010,Apotek SBS Farma Pandu Raya,Methycobal 500 mcg 10 Kapsul,Rp51.357,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150011,150012,Apotek SBS Farma Pandu Raya,Thiamycin 500 mg 10 Kapsul,Rp82.314,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150035,150036,Apotek SBS Farma Pandu Raya,Flunarizine 10 mg 10 Tablet - Obat Pereda Dema...,Rp26.997,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150052,150053,Apotek SBS Farma Pandu Raya,Cendo Catarlent Eye Drops 15 ml,Rp35.959,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150096,150097,Apotek SBS Farma Pandu Raya,Gliquidone 30 mg 10 Tablet,Rp18.416,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final2[~df_final2["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final2)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 150001
Total Images in Folder: 143438
Missing Images in Folder: 6563
Total Images & Missing Images in Folder 150001


In [ ]:
unavailable_ids[:5]

['150010', '150012', '150053', '150159', '150162']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
150009,150010,Apotek SBS Farma Pandu Raya,Methycobal 500 mcg 10 Kapsul,Rp51.357,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150011,150012,Apotek SBS Farma Pandu Raya,Thiamycin 500 mg 10 Kapsul,Rp82.314,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150052,150053,Apotek SBS Farma Pandu Raya,Cendo Catarlent Eye Drops 15 ml,Rp35.959,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150158,150159,Apotek SBS Farma Pandu Raya,Co Amoxiclav 625 mg 6 Tablet,Rp25.097,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991
150161,150162,Apotek SBS Farma Pandu Raya,Avodart 0.5 mg 10 Kapsul (strip),Rp140.358,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sbs-farma-pan...,14093991


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase2.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase2.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase2.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 3

In [ ]:
df_final3 = df_final.loc[300000:450000]
df_final3

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
300000,300001,Apotek Sultan Palembang by GoApotik,ANTANGIN JRG + MADU 15 ML BOX 12 SACHET,Rp4.508,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300001,300002,Apotek Sultan Palembang by GoApotik,CDR RASA JERUK TUBE 15 TABLET EFFERVESCENT,Rp86.972,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300002,300003,Apotek Sultan Palembang by GoApotik,BECOM-ZET STRIP 10 KAPLET,Rp37.385,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300003,300004,Apotek Sultan Palembang by GoApotik,BECOM-ZET STRIP 10 KAPLET,Rp36.296,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300004,300005,Apotek Sultan Palembang by GoApotik,PIL TUNTAS STRIP 10 KAPLET,Rp18.875,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
...,...,...,...,...,...,...,...,...
449996,449997,Apotek Tangor by GoApotik,DEXTAMINE STRIP 10 KAPLET,Rp28.383,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
449997,449998,Apotek Tangor by GoApotik,IMBOOST FORCE BOX 30 KAPLET,Rp309.624,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
449998,449999,Apotek Tangor by GoApotik,FATIGON STRIP ISI 6 KAPLET,Rp10.322,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
449999,450000,Apotek Tangor by GoApotik,LACTULOSE OGB DEXA MEDICA 3.3 GRAM/5 ML SYRUP ...,Rp45.154,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368


In [ ]:
df_todo = df_final3.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 150001/150001 [7:35:26<00:00,  5.49product/s, batch=301/301, Success: =139681, Fail: =10320]


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final3[~df_final3['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
300035,300036,Apotek Sultan Palembang by GoApotik,PROMAG HERBAL 10 ML SACHET,Rp3.289,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300117,300118,Apotek Sultan Palembang by GoApotik,VICKS FORMULA 44 ANAK RASA STRAWBERRY SIRUP 27 ML,Rp15.588,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300118,300119,Apotek Sultan Palembang by GoApotik,MINOSEP OBAT KUMUR MERAH 60 ML,Rp44.935,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300119,300120,Apotek Sultan Palembang by GoApotik,THROMBOPHOB GEL ISI 20 GRAM TUBE,Rp89.259,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300120,300121,Apotek Sultan Palembang by GoApotik,INSTO REGULAR DROPS 7.5 ML,Rp19.485,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
...,...,...,...,...,...,...,...,...
449809,449810,Vention Authorized Store,Vention [VAS-A16 3M] Kabel USB 2.0 Type A Male...,Rp34.295,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/sinshe-tekno/vention...,1727084
449831,449832,Apotek Tangor by GoApotik,C2FIT STRIP 4 KAPLET SALUT SELAPUT,Rp23.223,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
449843,449844,Apotek Tangor by GoApotik,ANLENE GOLD 5X RASA VANILA 850 GRAM BOX,Rp202.547,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
449847,449848,Apotek Tangor by GoApotik,SYNALTEN CREAM ISI 5 GRAM TUBE,Rp12.901,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368


In [ ]:
available_ids   = []
unavailable_ids = []

def check_image_available(row):
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    keyword      = ' '.join(str(product_name).split()[:3])
    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]
    try:
        res      = requests.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        return "ada" if products else "kosong"
    except:
        return "kosong"
rows = error_data.to_dict("records")

for row in tqdm(rows, unit="produk"):
    product_id = str(row["ID_Product"])
    status     = check_image_available(row)
    if status == "ada":
        available_ids.append(product_id)
    else:
        unavailable_ids.append(product_id)
total   = len(rows)
print(f"Total Available Images : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 10320/10320 [1:40:04<00:00,  1.72produk/s]

Total Available Images : 1,359 (13.2%)
Total Unavailable Images : 8,961 (86.8%)


In [ ]:
available_ids[:5]

['300456', '300597', '300699', '300750', '300808']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
300455,300456,Apotek Kartini by GoApotik,PARANERVION STRIP 10 KAPLET,Rp16.886,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-kartini-by-go...,15252103
300596,300597,Apotek Kartini by GoApotik,LELAP STRIP 4 KAPLET SALUT SELAPUT,Rp22.730,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kartini-by-go...,15252103
300698,300699,Apotek Japri Medan by GoApotik,ZINGOSERIN SYRUP ISI 60 ML BOTOL,Rp12.524,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-japri-medan-b...,15251937
300749,300750,Apotek Japri Medan by GoApotik,ZINGOSERIN SYRUP ISI 100 ML BOTOL,Rp18.185,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-japri-medan-b...,15251937
300807,300808,Apotek Japri Medan by GoApotik,DUN SALEP ISI 15 GRAM POT PLASTIK,Rp10.322,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-japri-medan-b...,15251937
...,...,...,...,...,...,...,...,...
449054,449055,Vention Authorized Store,Vention Casing 15 Plus Pro Max Magnetic Mags...,Rp139.125,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/sinshe-tekno/vention...,1727084
449282,449283,Vention Authorized Store,VENTION Car Phone Holder Air Vent Ultra Steady...,Rp135.315,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/sinshe-tekno/vention...,1727084
449559,449560,Vention Authorized Store,Vention Kabel Type C to HDMI Male Converter 4k...,Rp225.234,4.9,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/sinshe-tekno/vention...,1727084
449635,449636,Vention Authorized Store,Vention [HBC] Converter Mini DP to HDMI Pure,Rp144.530,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/sinshe-tekno/vention...,1727084


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 1359/1359 [04:22<00:00,  5.17product/s, batch=3/3, Success: =1332, Fail: =27]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-19:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 19/19 [00:03<00:00,  5.09product/s, batch=1/1, Success: =0, Fail: =19]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final3[~df_final3["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final3)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 150001
Total Images in Folder: 141021
Missing Images in Folder: 8980
Total Images & Missing Images in Folder 150001


In [ ]:
unavailable_ids[:5]

['300036', '300118', '300119', '300120', '300121']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
300035,300036,Apotek Sultan Palembang by GoApotik,PROMAG HERBAL 10 ML SACHET,Rp3.289,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300117,300118,Apotek Sultan Palembang by GoApotik,VICKS FORMULA 44 ANAK RASA STRAWBERRY SIRUP 27 ML,Rp15.588,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300118,300119,Apotek Sultan Palembang by GoApotik,MINOSEP OBAT KUMUR MERAH 60 ML,Rp44.935,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300119,300120,Apotek Sultan Palembang by GoApotik,THROMBOPHOB GEL ISI 20 GRAM TUBE,Rp89.259,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109
300120,300121,Apotek Sultan Palembang by GoApotik,INSTO REGULAR DROPS 7.5 ML,Rp19.485,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-sultan-palemb...,15041109


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase3.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase3.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase3.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 4

In [ ]:
df_final4 = df_final.loc[450000:600000]
df_final4

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
450000,450001,Apotek Tangor by GoApotik,METISOL 4 MG STRIP 10 TABLET,Rp6.451,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450001,450002,Apotek Tangor by GoApotik,RDL PAPAYA BRIGHTENING SOAP 135 GRAM,Rp20.643,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450002,450003,Apotek Tangor by GoApotik,CANDESARTAN CILEXETIL OGB BETA 8 MG STRIP 10 T...,Rp10.392,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450003,450004,Apotek Tangor by GoApotik,LACTACYD BABY BODY & HAIR WASH GENTLE CARE 150 ML,Rp103.208,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450004,450005,Apotek Tangor by GoApotik,ENERVON GOLD STRIP 5 KAPSUL,Rp28.383,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
...,...,...,...,...,...,...,...,...
599996,599997,Apotek Siwalan by GoApotik,SUTRA LUBRICANT 50 ML BOTOL,Rp25.980,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
599997,599998,Apotek Siwalan by GoApotik,PANTOPRAZOLE OGB DEXA MEDICA 40 MG BOX 30 TABLET,Rp350.677,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
599998,599999,Apotek Siwalan by GoApotik,TABLET TAMBAH DARAH PHAPROS BOX 100 TABLET,Rp77.928,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
599999,600000,Apotek Siwalan by GoApotik,PRIMPERAN 5 MG BOX 100 TABLET,Rp129.883,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206


In [ ]:
df_todo = df_final4.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 150001/150001 [10:36:14<00:00,  3.93product/s, batch=301/301, Success: =141395, Fail: =8606]


In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
drive.mount('/content/drive')
zip_path = "/content/drive/MyDrive/Data_Images1/images_product.zip"
destination_folder = "/content/images_product"
os.makedirs(destination_folder, exist_ok=True)
!unzip -q "{zip_path}" -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final4[~df_final4['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
450138,450139,Apotek Tangor by GoApotik,FIESTA KONDOM PARTY PACK BOX 12 PCS,Rp64.505,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450149,450150,Apotek Tangor by GoApotik,ETORICOXIB INTERBAT 90 MG STRIP 10 TABLET,Rp64.505,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450153,450154,Apotek Tangor by GoApotik,HANSAPLAST ANTISEPTIK PEMBERSIH LUKA 20 ML BOTOL,Rp19.352,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450192,450193,Apotek Tangor by GoApotik,Y-RINS 12 ML BOX ISI 2 BOTOL,Rp27.093,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450233,450234,Apotek Tangor by GoApotik,AZOVIR 5% CREAM ISI 5 GRAM TUBE,Rp21.933,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
...,...,...,...,...,...,...,...,...
599996,599997,Apotek Siwalan by GoApotik,SUTRA LUBRICANT 50 ML BOTOL,Rp25.980,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
599997,599998,Apotek Siwalan by GoApotik,PANTOPRAZOLE OGB DEXA MEDICA 40 MG BOX 30 TABLET,Rp350.677,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
599998,599999,Apotek Siwalan by GoApotik,TABLET TAMBAH DARAH PHAPROS BOX 100 TABLET,Rp77.928,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
599999,600000,Apotek Siwalan by GoApotik,PRIMPERAN 5 MG BOX 100 TABLET,Rp129.883,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206


In [ ]:
available_ids   = []
unavailable_ids = []

def check_image_available(row):
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = requests.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        return "ada" if products else "kosong"
    except:
        return "kosong"
rows = error_data.to_dict("records")

for row in tqdm(rows, unit="produk"):
    product_id = str(row["ID_Product"])
    status     = check_image_available(row)
    if status == "ada":
        available_ids.append(product_id)
    else:
        unavailable_ids.append(product_id)
total   = len(rows)
print(f"Total Available Images : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 8606/8606 [1:24:56<00:00,  1.69produk/s]

Total Available Images : 561 (6.5%)
Total Unavailable Images : 8,045 (93.5%)


In [ ]:
available_ids[:5]

['450150', '450154', '450193', '450234', '450260']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
450149,450150,Apotek Tangor by GoApotik,ETORICOXIB INTERBAT 90 MG STRIP 10 TABLET,Rp64.505,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450153,450154,Apotek Tangor by GoApotik,HANSAPLAST ANTISEPTIK PEMBERSIH LUKA 20 ML BOTOL,Rp19.352,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450192,450193,Apotek Tangor by GoApotik,Y-RINS 12 ML BOX ISI 2 BOTOL,Rp27.093,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450233,450234,Apotek Tangor by GoApotik,AZOVIR 5% CREAM ISI 5 GRAM TUBE,Rp21.933,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450259,450260,Apotek Tangor by GoApotik,L-VIT D3 400 IU/0.5 ML DROP 15 ML,Rp74.827,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
...,...,...,...,...,...,...,...,...
599375,599376,Apotek Siwalan by GoApotik,CLOPIDOGREL BISULFATE PHAPROS 75 MG BOX 30 TABLET,Rp121.771,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
599384,599385,Apotek Siwalan by GoApotik,CLOPIDOGREL ETERCON 75 MG STRIP 10 TABLET,Rp12.658,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
599386,599387,Apotek Siwalan by GoApotik,CLOPIDOGREL BISULFATE PHAPROS 75 MG BOX 30 TABLET,Rp115.076,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
599510,599511,Apotek Siwalan by GoApotik,ROSUFER 10 MG BOX 30 TABLET,Rp459.627,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 561/561 [02:25<00:00,  3.86product/s, batch=2/2, Success: =558, Fail: =3]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-3:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 3/3 [00:02<00:00,  1.41product/s, batch=1/1, Success: =0, Fail: =3]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final4[~df_final4["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final4)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 150001
Total Images in Folder: 141953
Missing Images in Folder: 8048
Total Images & Missing Images in Folder 150001


In [ ]:
unavailable_ids[:5]

['450139', '450276', '450489', '450695', '450711']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
450138,450139,Apotek Tangor by GoApotik,FIESTA KONDOM PARTY PACK BOX 12 PCS,Rp64.505,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450275,450276,Apotek Tangor by GoApotik,MY BABY DIAPER RASH CREAM,Rp19.352,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450488,450489,Apotek Tangor by GoApotik,FRESH CARE SANDALWOOD BOTOL ROLL ON 10 ML,Rp16.886,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450694,450695,Apotek Tangor by GoApotik,SANMOL FORTE SYRUP 60 ML BOTOL,Rp51.604,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368
450710,450711,Apotek Tangor by GoApotik,JOHNSON'S BABY BLOSSOMS SOAP 100 GRAM BOX,Rp9.093,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-tangor-by-goa...,15047368


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase4.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase4.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase4.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 5

In [ ]:
df_final5 = df_final.loc[600000:800000]
df_final5

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
600000,600001,Apotek Siwalan by GoApotik,CENDO AUGENTONIC 15 ML TETES MATA,Rp41.564,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600001,600002,Apotek Siwalan by GoApotik,METFORMIN PROMED 500 MG STRIP 10 KAPLET,Rp12.988,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600002,600003,Apotek Siwalan by GoApotik,GABAPENTIN OGB DEXA MEDICA 100 MG STRIP 10 KAPSUL,Rp25.976,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600003,600004,Apotek Siwalan by GoApotik,IKA SARIAWAN LARUTAN 120 ML,Rp23.380,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600004,600005,Apotek Siwalan by GoApotik,MECOBALAMIN LAPI 500 MCG BOX 100 KAPSUL,Rp142.868,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
...,...,...,...,...,...,...,...,...
799996,799997,Apotek Kama Farma by GoApotik,RHEU-TREX 2.5 MG BOX 50 TABLET,Rp133.340,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
799997,799998,Apotek Kama Farma by GoApotik,FARLOSIN SR 0.4 MG BOX 30 TABLET,Rp266.677,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
799998,799999,Apotek Kama Farma by GoApotik,LANCID 30 MG BOX 20 KAPSUL,Rp412.800,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
799999,800000,Apotek Kama Farma by GoApotik,NISTROL 20 MG STRIP 10 TABLET,Rp96.807,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483


In [ ]:
df_todo = df_final5.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 200001/200001 [5:37:00<00:00,  9.89product/s, batch=401/401, Success: =187957, Fail: =12044]


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final5[~df_final5['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
600000,600001,Apotek Siwalan by GoApotik,CENDO AUGENTONIC 15 ML TETES MATA,Rp41.564,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600001,600002,Apotek Siwalan by GoApotik,METFORMIN PROMED 500 MG STRIP 10 KAPLET,Rp12.988,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600003,600004,Apotek Siwalan by GoApotik,IKA SARIAWAN LARUTAN 120 ML,Rp23.380,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600004,600005,Apotek Siwalan by GoApotik,MECOBALAMIN LAPI 500 MCG BOX 100 KAPSUL,Rp142.868,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600005,600006,Apotek Siwalan by GoApotik,BEBITHEN 5% CREAM 20 GRAM TUBE,Rp58.448,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
...,...,...,...,...,...,...,...,...
799850,799851,Apotek Kama Farma by GoApotik,MUCOS SYRUP ISI 60 ML BOTOL,Rp20.703,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
799912,799913,Apotek Kama Farma by GoApotik,BUDESMA 0.5 MG/ML INHALER 2 ML CONTAINER MONODOSE,Rp144.299,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
799929,799930,Apotek Kama Farma by GoApotik,ANGIOTEN 50 MG STRIP 10 TABLET,Rp124.816,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
799930,799931,Apotek Kama Farma by GoApotik,ANGIOTEN 50 MG BOX 30 TABLET,Rp381.140,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483


In [ ]:
available_ids   = []
unavailable_ids = []

def check_image_available(row):
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = requests.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        return "ada" if products else "kosong"
    except:
        return "kosong"

rows = error_data.to_dict("records")

for row in tqdm(rows, unit="produk"):
    product_id = str(row["ID_Product"])
    status     = check_image_available(row)
    if status == "ada":
        available_ids.append(product_id)
    else:
        unavailable_ids.append(product_id)
total   = len(rows)
print(f"Total Available Images : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 12044/12044 [50:37<00:00,  3.97produk/s]

Total Available Images : 1,241 (10.3%)
Total Unavailable Images : 10,803 (89.7%)


In [ ]:
available_ids[:5]

['600211', '600360', '600441', '600596', '600854']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
600210,600211,Apotek Habibi by GoApotik,EDOTIN SYRUP ISI 60 ML BOTOL,Rp52.972,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/ahabibi/edotin-syrup...,9776563
600359,600360,Apotek Habibi by GoApotik,CURVINO KIDS SYRUP ISI 60 ML BOTOL,Rp178.516,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/ahabibi/curvino-kids...,9776563
600440,600441,Apotek Habibi by GoApotik,FLAMAR 25 MG BOX 100 TABLET,Rp147.196,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/ahabibi/flamar-25-mg...,9776563
600595,600596,Apotek Habibi by GoApotik,MAXPOFER BOX 10 TABLET EFFERVESENT,Rp170.478,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/ahabibi/maxpofer-box...,9776563
600853,600854,Apotek Dua Putri Tangerang,Apolar N Cream 10 g (tube),Rp59.540,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotekduaputritgr/ap...,14827545
...,...,...,...,...,...,...,...,...
799364,799365,Apotek Kama Farma by GoApotik,DERMOVEL 0.1% KRIM 10 GRAM,Rp87.675,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
799365,799366,Apotek Kama Farma by GoApotik,DERMOVEL 0.1% KRIM 5 GRAM,Rp55.895,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
799369,799370,Apotek Kama Farma by GoApotik,EZEFER 10 MG STRIP 10 TABLET,Rp142.473,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
799703,799704,Apotek Kama Farma by GoApotik,CARVILOL 25 MG STRIP 10 TABLET,Rp73.063,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 1241/1241 [02:05<00:00,  9.88product/s, batch=3/3, Success: =1228, Fail: =13]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-2:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 2/2 [00:01<00:00,  1.60product/s, batch=1/1, Success: =0, Fail: =2]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final5[~df_final5["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final5)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 200001
Total Images in Folder: 189196
Missing Images in Folder: 10805
Total Images & Missing Images in Folder 200001


In [ ]:
unavailable_ids[:5]

['600001', '600002', '600004', '600005', '600006']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
600000,600001,Apotek Siwalan by GoApotik,CENDO AUGENTONIC 15 ML TETES MATA,Rp41.564,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600001,600002,Apotek Siwalan by GoApotik,METFORMIN PROMED 500 MG STRIP 10 KAPLET,Rp12.988,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600003,600004,Apotek Siwalan by GoApotik,IKA SARIAWAN LARUTAN 120 ML,Rp23.380,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600004,600005,Apotek Siwalan by GoApotik,MECOBALAMIN LAPI 500 MCG BOX 100 KAPSUL,Rp142.868,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206
600005,600006,Apotek Siwalan by GoApotik,BEBITHEN 5% CREAM 20 GRAM TUBE,Rp58.448,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-siwalan-by-go...,7219206


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase5.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase5.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase5.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 6

In [ ]:
df_final6 = df_final.loc[800000:1000000]
df_final6

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
800000,800001,Apotek Kama Farma by GoApotik,CENDO XITROL TETES MATA 5 ML,Rp40.795,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800001,800002,Apotek Kama Farma by GoApotik,AMADIAB 2 MG STRIP 10 KAPLET,Rp65.148,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800002,800003,Apotek Kama Farma by GoApotik,XENICAL 120 MG BOX 21 KAPSUL,Rp316.602,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800003,800004,Apotek Kama Farma by GoApotik,CAL-95 STRIP ISI 6 KAPLET,Rp55.408,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800004,800005,Apotek Kama Farma by GoApotik,NATERAN 25 MG STRIP 10 TABLET,Rp529.699,4.9,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
...,...,...,...,...,...,...,...,...
999996,999997,Apotek Meri Medica by GoApotik,MAXPOFER BOX 10 TABLET EFFERVESENT,Rp161.346,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
999997,999998,Apotek Meri Medica by GoApotik,GOOD LIFE GCM BOX 30 KAPLET,Rp286.161,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
999998,999999,Apotek Meri Medica by GoApotik,ETORICOXIB NOVELL 60 MG STRIP 10 TABLET,Rp56.624,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
999999,1000000,Apotek Meri Medica by GoApotik,ELZSA PIL KB BOX 21 TABLET,Rp158.181,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847


In [ ]:
df_todo = df_final6.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 200001/200001 [6:39:19<00:00,  8.35product/s, batch=401/401, Success: =186772, Fail: =13229]


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final6[~df_final6['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
800068,800069,Apotek Kama Farma by GoApotik,TROLIP 160 MG STRIP 6 TABLET,Rp103.507,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800086,800087,Apotek Kama Farma by GoApotik,TENSIVASK 5 MG STRIP 10 TABLET,Rp89.400,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800124,800125,Apotek Kama Farma by GoApotik,PREGABALIN OGB DEXA MEDICA 150 MG BOX 30 KAPSUL,Rp300.774,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800127,800128,Apotek Kama Farma by GoApotik,RANITIDINE HCL OGB DEXA MEDICA 150 MG BOX 100 ...,Rp39.576,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800162,800163,Apotek Kama Farma by GoApotik,STALEVO BOX ISI 30 TABLET,Rp441.186,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
...,...,...,...,...,...,...,...,...
999785,999786,Apotek Meri Medica by GoApotik,SOFFELL LOTION BUNGA GERANIUM 60 GRAM BOTOL,Rp17.658,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
999989,999990,Apotek Meri Medica by GoApotik,POLAR BEAR MINYAK ANGIN 27 ML,Rp51.769,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
999990,999991,Apotek Meri Medica by GoApotik,POLAR BEAR MINYAK ANGIN 18 ML,Rp37.981,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
999991,999992,Apotek Meri Medica by GoApotik,POLAR BEAR MINYAK ANGIN 8 ML,Rp26.490,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847


In [ ]:
available_ids   = []
unavailable_ids = []

def check_image_available(row):
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = requests.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        return "ada" if products else "kosong"
    except:
        return "kosong"

rows = error_data.to_dict("records")

for row in tqdm(rows, unit="produk"):
    product_id = str(row["ID_Product"])
    status     = check_image_available(row)
    if status == "ada":
        available_ids.append(product_id)
    else:
        unavailable_ids.append(product_id)
total   = len(rows)
print(f"Total Available Images : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 13229/13229 [43:31<00:00,  5.07produk/s]

Total Available Images : 788 (6.0%)
Total Unavailable Images : 12,441 (94.0%)


In [ ]:
available_ids[:5]

['800087', '800125', '800247', '800267', '800585']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
800086,800087,Apotek Kama Farma by GoApotik,TENSIVASK 5 MG STRIP 10 TABLET,Rp89.400,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800124,800125,Apotek Kama Farma by GoApotik,PREGABALIN OGB DEXA MEDICA 150 MG BOX 30 KAPSUL,Rp300.774,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800246,800247,Apotek Farmarin Kota Malang by GoApotik,HYLOQUIN STRIP ISI 10 TABLET,Rp295.353,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-farmarin-kota...,18031112
800266,800267,Apotek Farmarin Kota Malang by GoApotik,ULSAFATE 500 MG STRIP 10 TABLET,Rp37.591,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-farmarin-kota...,18031112
800584,800585,Apotek Abdusaalam Almadani by GoApotik,ANDALAN BLISTER 28 TABLET,Rp10.093,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-abdusaalam-al...,16437261
...,...,...,...,...,...,...,...,...
999043,999044,Apotek Meri Medica by GoApotik,PROVE D3 DROPS ISI 12.5 ML BOTOL,Rp356.786,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
999046,999047,Apotek Meri Medica by GoApotik,PARAMEX NYERI OTOT STRIP 4 TABLET,Rp4.021,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
999069,999070,Apotek Meri Medica by GoApotik,CRESTOR 20 MG STRIP 15 TABLET,Rp650.982,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
999070,999071,Apotek Meri Medica by GoApotik,RECO EYE DROPS 10 ML BOTOL,Rp14.613,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 788/788 [01:32<00:00,  8.51product/s, batch=2/2, Success: =786, Fail: =2]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-1*error_count:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 2/2 [00:01<00:00,  1.72product/s, batch=1/1, Success: =0, Fail: =2]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final6[~df_final6["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final6)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 200001
Total Images in Folder: 187558
Missing Images in Folder: 12443
Total Images & Missing Images in Folder 200001


In [ ]:
unavailable_ids[:5]

['800069', '800128', '800163', '800334', '800398']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
800068,800069,Apotek Kama Farma by GoApotik,TROLIP 160 MG STRIP 6 TABLET,Rp103.507,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800127,800128,Apotek Kama Farma by GoApotik,RANITIDINE HCL OGB DEXA MEDICA 150 MG BOX 100 ...,Rp39.576,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800162,800163,Apotek Kama Farma by GoApotik,STALEVO BOX ISI 30 TABLET,Rp441.186,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-kama-farma-by...,16854483
800333,800334,Apotek Farmarin Kota Malang by GoApotik,CAMELOC 15 MG STRIP 10 TABLET,Rp32.232,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-farmarin-kota...,18031112
800397,800398,Apotek Farmarin Kota Malang by GoApotik,TIRIZ 10 MG STRIP 10 TABLET,Rp85.030,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-farmarin-kota...,18031112


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase6.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase6.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase6.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 7

In [ ]:
df_final7 = df_final.loc[1000000:1250000]
df_final7

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1000000,1000001,Apotek Meri Medica by GoApotik,EUPHYLLIN RETARD MITE 125 MG STRIP 10 TABLET,Rp29.835,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000001,1000002,Apotek Meri Medica by GoApotik,SOFFELL LOTION BUNGA GERAN 80 GRAM,Rp16.440,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000002,1000003,Apotek Meri Medica by GoApotik,SOFFELL KULIT JERUK 80 ML,Rp17.658,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000003,1000004,Apotek Meri Medica by GoApotik,FLAMICORT 4 MG STRIP 10 TABLET,Rp56.624,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000004,1000005,Apotek Meri Medica by GoApotik,FAMOCID 40 MG STRIP 6 TABLET,Rp60.765,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
...,...,...,...,...,...,...,...,...
1249996,1249997,Apotek Ciatra Bekasi by GoApotik,VIDON 10 MG STRIP 10 TABLET,Rp8.444,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1249997,1249998,Apotek Ciatra Bekasi by GoApotik,NEUROSANTIN 300 MG BOX 50 KAPSUL,Rp486.776,4.8,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1249998,1249999,Apotek Ciatra Bekasi by GoApotik,NOROID SOOTHING LOTION 200 ML TUBE,Rp237.031,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1249999,1250000,Apotek Ciatra Bekasi by GoApotik,MEDERMA PROAKTIF GEL 20 GRAM,Rp295.478,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884


In [ ]:
df_todo = df_final7.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 250001/250001 [7:15:28<00:00,  9.57product/s, batch=501/501, Success: =232485, Fail: =17120]


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final7[~df_final7['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1000001,1000002,Apotek Meri Medica by GoApotik,SOFFELL LOTION BUNGA GERAN 80 GRAM,Rp16.440,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000002,1000003,Apotek Meri Medica by GoApotik,SOFFELL KULIT JERUK 80 ML,Rp17.658,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000040,1000041,Apotek Meri Medica by GoApotik,KILLBAC WOUND IRRIGATION SOLUTION 350 ML,Rp239.888,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000063,1000064,Apotek Meri Medica by GoApotik,PAGODA SALEP EKSTRA 10 GRAM POT,Rp10.717,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000102,1000103,Apotek Meri Medica by GoApotik,SELECTION KAPAS BOX 50 GRAM,Rp15.831,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
...,...,...,...,...,...,...,...,...
1249932,1249933,Apotek Ciatra Bekasi by GoApotik,GLIQUIDONE OGB DEXA MEDICA 30 MG STRIP 10 TABL...,Rp20.702,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1249933,1249934,Apotek Ciatra Bekasi by GoApotik,ERAPHAGE 500 MG BOX 100 KAPLET,Rp259.613,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1249934,1249935,Apotek Ciatra Bekasi by GoApotik,ERAPHAGE 500 MG STRIP 10 KAPLET,Rp29.208,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1249949,1249950,Apotek Ciatra Bekasi by GoApotik,ESTALEX 50 MG STRIP 10 TABLET,Rp25.976,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884


In [ ]:
available_ids   = []
unavailable_ids = []

def check_image_available(row):
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = requests.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        return "ada" if products else "kosong"
    except:
        return "kosong"

rows = error_data.to_dict("records")

for row in tqdm(rows, unit="produk"):
    product_id = str(row["ID_Product"])
    status     = check_image_available(row)
    if status == "ada":
        available_ids.append(product_id)
    else:
        unavailable_ids.append(product_id)
total   = len(rows)
print(f"Total Available Images : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 17120/17120 [58:12<00:00,  4.90produk/s]

Total Available Images : 1,206 (7.0%)
Total Unavailable Images : 15,914 (93.0%)


In [ ]:
available_ids[:5]

['1000255', '1000256', '1000257', '1000749', '1000779']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1000254,1000255,Apotek Meri Medica by GoApotik,EASY TOUCH URIC ACID TEST STRIP BOX 25 PCS,Rp132.730,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000255,1000256,Apotek Meri Medica by GoApotik,"LACTULAX 3,335 G / 5 ML SIRUP 60 ML",Rp78.378,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000256,1000257,Apotek Meri Medica by GoApotik,"LACTULAX 3,335 G / 5 ML SIRUP 120 ML",Rp107.166,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000748,1000749,Apotek Meri Medica by GoApotik,DARYANT-TULLE KASSA STERIL 10 X 10 CM 1 PIECE,Rp38.967,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000778,1000779,Apotek Meri Medica by GoApotik,CENDO VITROLENTA TETES MATA 5 ML,Rp53.581,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
...,...,...,...,...,...,...,...,...
1249403,1249404,Apotek Ciatra Bekasi by GoApotik,DULCOLAX ANAK 5 MG SUPPOSITORIA,Rp31.822,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1249460,1249461,Apotek Ciatra Bekasi by GoApotik,NITROKAF RETARD FORTE BOX 100 KAPSUL,Rp421.872,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1249932,1249933,Apotek Ciatra Bekasi by GoApotik,GLIQUIDONE OGB DEXA MEDICA 30 MG STRIP 10 TABL...,Rp20.702,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1249933,1249934,Apotek Ciatra Bekasi by GoApotik,ERAPHAGE 500 MG BOX 100 KAPLET,Rp259.613,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 1206/1206 [02:13<00:00,  9.05product/s, batch=3/3, Success: =1200, Fail: =6]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-6:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 6/6 [00:01<00:00,  4.77product/s, batch=1/1, Success: =0, Fail: =6]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final7[~df_final7["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final7)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 250001
Total Images in Folder: 234081
Missing Images in Folder: 15920
Total Images & Missing Images in Folder 250001


In [ ]:
unavailable_ids[:5]

['1000002', '1000003', '1000041', '1000064', '1000103']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1000001,1000002,Apotek Meri Medica by GoApotik,SOFFELL LOTION BUNGA GERAN 80 GRAM,Rp16.440,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000002,1000003,Apotek Meri Medica by GoApotik,SOFFELL KULIT JERUK 80 ML,Rp17.658,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000040,1000041,Apotek Meri Medica by GoApotik,KILLBAC WOUND IRRIGATION SOLUTION 350 ML,Rp239.888,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000063,1000064,Apotek Meri Medica by GoApotik,PAGODA SALEP EKSTRA 10 GRAM POT,Rp10.717,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847
1000102,1000103,Apotek Meri Medica by GoApotik,SELECTION KAPAS BOX 50 GRAM,Rp15.831,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-meri-medica-b...,12887847


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase7.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase7.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase7.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 8

In [ ]:
drive.mount('/content/drive')
zip_path = "/content/drive/MyDrive/Data_Images1/images_product.zip"
destination_folder = "/content/images_product"
os.makedirs(destination_folder, exist_ok=True)
!unzip -q "{zip_path}" -d /content/

Mounted at /content/drive


In [ ]:
df_final8 = df_final.loc[1250000:1500000]
df_final8

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1250000,1250001,Apotek Ciatra Bekasi by GoApotik,NEO HP PRO STRIP 10 KAPSUL,Rp115.683,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250001,1250002,Apotek Ciatra Bekasi by GoApotik,NAPREX SUSPENSI ISI 60 ML BOTOL,Rp59.747,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250002,1250003,Apotek Ciatra Bekasi by GoApotik,LEVETIRACETAM AMAROX 250 MG BOX 30 TABLET,Rp194.823,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250003,1250004,Apotek Ciatra Bekasi by GoApotik,LEVETIRACETAM AMAROX 250 MG STRIP 10 TABLET,Rp64.943,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250004,1250005,Apotek Ciatra Bekasi by GoApotik,ORLIBES 120 MG BOX 30 KAPSUL,Rp249.629,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
...,...,...,...,...,...,...,...,...
1499996,1499997,Bonjour Electronic Official,Kulkas LG Smart Inverter 2 Pintu GN-C702HQCL 5...,Rp9.999.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1499997,1499998,Bonjour Electronic Official,Mesin Cuci 10 Kg Front Loading Sharp ES-FL1410...,Rp6.990.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1499998,1499999,Bonjour Electronic Official,Mesin Cuci 9 Kg Front Loading Sharp ES-FL1490M...,Rp6.740.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1499999,1500000,Bonjour Electronic Official,Mesin Cuci 7Kg Front Loading Sharp ES-FL1270MW...,Rp4.491.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313


In [ ]:
done_ids = set(f.replace(".jpg", "")for f in os.listdir(path_folder)if f.endswith(".jpg"))
df_todo = df_final8[~df_final8["ID_Product"].astype(str).isin(done_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 83115/83115 [4:31:37<00:00,  5.10product/s, batch=167/167, Success: =72232, Fail: =10883]


In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final8[~df_final8['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1250037,1250038,Apotek Ciatra Bekasi by GoApotik,RYDIAN 10 MG STRIP 10 TABLET,Rp47.814,4.8,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250217,1250218,Apotek Ciatra Bekasi by GoApotik,LINOVEN 5 MG BOX 30 TABLET,Rp129.880,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250224,1250225,Apotek Ciatra Bekasi by GoApotik,DERMATIX ULTRA GEL TUBE 9 GRAM,Rp246.772,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250236,1250237,Apotek Ciatra Bekasi by GoApotik,ASTINA 6 MG BOX 30 KAPSUL,Rp272.748,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250297,1250298,Apotek Ciatra Bekasi by GoApotik,INCETYL 200 MG BOX 60 KAPSUL,Rp121.772,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
...,...,...,...,...,...,...,...,...
1499179,1499180,Bonjour Electronic Official,QLED TV Sharp 4T-C55HL6500i 4K 55 Inch Google ...,Rp9.290.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1499180,1499181,Bonjour Electronic Official,QLED TV Sharp 4T-C50HL6500i 4K 50 Inch Google ...,Rp8.290.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1499290,1499291,Bonjour Electronic Official,Speaker Bluetooth Polytron PAS-8E20-M/B Extra ...,Rp1.690.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1499763,1499764,Bonjour Electronic Official,LED TV 50 Inch 4K Smart TV 50 Hisense 50E6K 50...,Rp5.390.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313


In [ ]:
available_ids   = []
unavailable_ids = []

def check_image_available(row):
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = requests.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        return "ada" if products else "kosong"
    except:
        return "kosong"

rows = error_data.to_dict("records")

for row in tqdm(rows, unit="produk"):
    product_id = str(row["ID_Product"])
    status     = check_image_available(row)
    if status == "ada":
        available_ids.append(product_id)
    else:
        unavailable_ids.append(product_id)
total   = len(rows)
print(f"Total Available Images : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 10883/10883 [1:34:59<00:00,  1.91produk/s]

Total Available Images : 178 (1.6%)
Total Unavailable Images : 10,705 (98.4%)


In [ ]:
available_ids[:5]

['1254035', '1265943', '1279581', '1280094', '1285129']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1254034,1254035,Apotek Humairah Farma Bandung by GoApotik,BEDAK SALICYL KIMIA FARMA BOX 60 GRAM,Rp9.214,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-humairah-farm...,16186576
1265942,1265943,Apotek Mose Pamulang by GoA,HOLLY SABUN HIJAU ANTISEPTIK 60 GRAM,Rp7.796,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/amosepmlgoa/holly-sa...,17871523
1279580,1279581,Apotek Wira Lestari by GoApotik,RENOVIT BOTOL ISI 30 KAPLET,Rp114.908,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-wira-lestari-...,6507175
1280093,1280094,Apotek Intan Sehat Medika by GoApotik,BIO LIFE AHFC 3.5 GRAM SACHET,Rp29.226,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-intan-sehat-m...,13403563
1285128,1285129,Apotek Vaza Farma by GoApotik,PLOSSA PRESS AND SOOTHE AROMATICS EUCALYPTUS,Rp15.023,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-vaza-farma-by...,9472692
...,...,...,...,...,...,...,...,...
1499179,1499180,Bonjour Electronic Official,QLED TV Sharp 4T-C55HL6500i 4K 55 Inch Google ...,Rp9.290.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1499180,1499181,Bonjour Electronic Official,QLED TV Sharp 4T-C50HL6500i 4K 50 Inch Google ...,Rp8.290.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1499290,1499291,Bonjour Electronic Official,Speaker Bluetooth Polytron PAS-8E20-M/B Extra ...,Rp1.690.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1499763,1499764,Bonjour Electronic Official,LED TV 50 Inch 4K Smart TV 50 Hisense 50E6K 50...,Rp5.390.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 178/178 [00:43<00:00,  4.11product/s, batch=1/1, Success: =173, Fail: =5]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-1*error_count:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 5/5 [00:01<00:00,  2.90product/s, batch=1/1, Success: =0, Fail: =5]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final8[~df_final8["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final8)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 250001
Total Images in Folder: 239291
Missing Images in Folder: 10710
Total Images & Missing Images in Folder 250001


In [ ]:
unavailable_ids[:5]

['1250038', '1250218', '1250225', '1250237', '1250298']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1250037,1250038,Apotek Ciatra Bekasi by GoApotik,RYDIAN 10 MG STRIP 10 TABLET,Rp47.814,4.8,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250217,1250218,Apotek Ciatra Bekasi by GoApotik,LINOVEN 5 MG BOX 30 TABLET,Rp129.880,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250224,1250225,Apotek Ciatra Bekasi by GoApotik,DERMATIX ULTRA GEL TUBE 9 GRAM,Rp246.772,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250236,1250237,Apotek Ciatra Bekasi by GoApotik,ASTINA 6 MG BOX 30 KAPSUL,Rp272.748,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884
1250297,1250298,Apotek Ciatra Bekasi by GoApotik,INCETYL 200 MG BOX 60 KAPSUL,Rp121.772,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ciatra-bekasi...,13806884


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase8.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase8.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase8.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 9

In [ ]:
drive.mount('/content/drive')
zip_path = "/content/drive/MyDrive/Data_Images1/images_product.zip"
destination_folder = "/content/images_product"
os.makedirs(destination_folder, exist_ok=True)
!unzip -q "{zip_path}" -d /content/

Mounted at /content/drive


In [ ]:
df_final9 = df_final.loc[1500000:1750000]
df_final9

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1500000,1500001,Bonjour Electronic Official,Speaker Polytron PAS PRO15F3 Speaker Bluetooth...,Rp3.391.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1500001,1500002,Bonjour Electronic Official,TV Samsung 85DU7000 4K UHD Samsung Smart TV 85...,Rp22.991.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1500002,1500003,Bonjour Electronic Official,TV Samsung 75DU7000 4K UHD Samsung Smart TV 75...,Rp11.999.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1500003,1500004,Bonjour Electronic Official,TV Samsung 70DU7000 4K UHD Samsung Smart TV 70...,Rp9.750.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
1500004,1500005,Bonjour Electronic Official,TV Samsung 65DU7000 4K UHD Samsung Smart TV 65...,Rp7.990.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/bonjour-electronic-o...,9038313
...,...,...,...,...,...,...,...,...
1749996,1749997,Vention Mobile & Comp,Vention Kabel AUX XLR Audio Extension Cable Ma...,Rp66.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-k...,13319967
1749997,1749998,Vention Mobile & Comp,Vention Kabel Audio AUX 6.5mm TRS Male To XLR ...,Rp68.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-k...,13319967
1749998,1749999,Vention Mobile & Comp,Vention Kabel Audio AUX 6.5mm TRS Male To XLR ...,Rp150.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-k...,13319967
1749999,1750000,Vention Mobile & Comp,Vention Kabel Audio Cable AUX 3.5mm TRRS to 6....,Rp92.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-k...,13319967


In [ ]:
done_ids = set(f.replace(".jpg", "")for f in os.listdir(path_folder)if f.endswith(".jpg"))
df_todo = df_final9[~df_final9["ID_Product"].astype(str).isin(done_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 47539/47539 [2:30:09<00:00,  5.28product/s, batch=96/96, Success: =35046, Fail: =12493]


In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final9[~df_final9['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1501168,1501169,balmgalery,[TIDAK UNTUK DIJUAL] Jedai Original Bangkok 5 cm,Rp1.000.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/balmgalery/tidak-unt...,531481
1501178,1501179,balmgalery,[FREE GIFT] Gantungan Tassel Kipas (Spesial CNY),Rp992.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/balmgalery/free-gift...,531481
1501186,1501187,balmgalery,[FREE GIFT] POUCH TOFU MESH,Rp1.000.000,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/balmgalery/free-gift...,531481
1501302,1501303,Apotek Jodipati by GoApotik,MADU ENAK STICK JERUK 15 GRAM SACHET,Rp1.906,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-jodipati-by-g...,18029783
1501305,1501306,Apotek Jodipati by GoApotik,VERILE ACNE GEL 10 GRAM TUBE,Rp21.600,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-jodipati-by-g...,18029783
...,...,...,...,...,...,...,...,...
1749679,1749680,JDM CLOTHING,JDMCLOTHING - GAMIS RHINESTONE LIST REFLECTIVE...,Rp194.999,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/jdmclothing/jdmcloth...,9054944
1749718,1749719,Vention Mobile & Comp,Vention Universal Waterproof Handphone Pouch S...,Rp45.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/ventionacc/vention-u...,13319967
1749778,1749779,Vention Mobile & Comp,Vention Kabel Portable HDMI 2.0 Male to Cable ...,Rp40.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-k...,13319967
1749849,1749850,Vention Mobile & Comp,Vention Kabel Colokan C7 Connector Power Cord ...,Rp35.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-k...,13319967


In [ ]:
available_ids   = []
unavailable_ids = []

def check_image_available(row):
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = requests.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        return "ada" if products else "kosong"
    except:
        return "kosong"
rows = error_data.to_dict("records")

for row in tqdm(rows, unit="produk"):
    product_id = str(row["ID_Product"])
    status     = check_image_available(row)
    if status == "ada":
        available_ids.append(product_id)
    else:
        unavailable_ids.append(product_id)
total   = len(rows)
print(f"Total Available Images : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 12493/12493 [1:43:11<00:00,  2.02produk/s]

Total Available Images : 171 (1.4%)
Total Unavailable Images : 12,322 (98.6%)


In [ ]:
available_ids[:5]

['1527483', '1551066', '1551075', '1551121', '1551164']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1527482,1527483,Apotek Nufa by GoApotik,IMBOOST STRIP ISI 4 TABLET,Rp25.802,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-nufa-by-goapo...,17871497
1551065,1551066,Apotek Citra Medika Plaju by GoApotik,ACYCLOVIR INDOFARMA KRIM 5 GRAM,Rp10.392,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-citra-medika-...,16684347
1551074,1551075,Apotek Citra Medika Plaju by GoApotik,ACICLOVIR SAMPHARINDO 400 MG STRIP 10 TABLET,Rp14.192,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-citra-medika-...,16684347
1551120,1551121,Apotek Citra Medika Plaju by GoApotik,GUAIFENESIN IMFARMIND 100 MG STRIP 10 TABLET,Rp6.495,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-citra-medika-...,16684347
1551163,1551164,Apotek Citra Medika Plaju by GoApotik,KONVERMEX 250 MG / 5 ML SUSPENSI 10 ML,Rp27.093,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-citra-medika-...,16684347
...,...,...,...,...,...,...,...,...
1748001,1748002,United Cleaning Official,[BUNDLE] Hand Sanitizer Cair 70% Food Grade 10...,Rp50.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/united-cleaning-offi...,1462307
1748180,1748181,United Cleaning Official,[BUNDLE] Hand Sanitizer GEL 70% Food Grade PRO...,Rp720.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/united-cleaning-offi...,1462307
1748181,1748182,United Cleaning Official,[BUNDLE] Hand Sanitizer GEL 70% Food Grade PRO...,Rp75.000,4.8,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/united-cleaning-offi...,1462307
1748182,1748183,United Cleaning Official,[BUNDLE] Hand Sanitizer CAIR 70% Food Grade PR...,Rp480.000,4.6,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/united-cleaning-offi...,1462307


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 171/171 [00:32<00:00,  5.24product/s, batch=1/1, Success: =171, Fail: =0]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final9[~df_final9["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final9)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 250001
Total Images in Folder: 237679
Missing Images in Folder: 12322
Total Images & Missing Images in Folder 250001


In [ ]:
unavailable_ids[:5]

['1501169', '1501179', '1501187', '1501303', '1501306']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1501168,1501169,balmgalery,[TIDAK UNTUK DIJUAL] Jedai Original Bangkok 5 cm,Rp1.000.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/balmgalery/tidak-unt...,531481
1501178,1501179,balmgalery,[FREE GIFT] Gantungan Tassel Kipas (Spesial CNY),Rp992.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/balmgalery/free-gift...,531481
1501186,1501187,balmgalery,[FREE GIFT] POUCH TOFU MESH,Rp1.000.000,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/balmgalery/free-gift...,531481
1501302,1501303,Apotek Jodipati by GoApotik,MADU ENAK STICK JERUK 15 GRAM SACHET,Rp1.906,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-jodipati-by-g...,18029783
1501305,1501306,Apotek Jodipati by GoApotik,VERILE ACNE GEL 10 GRAM TUBE,Rp21.600,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-jodipati-by-g...,18029783


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase9.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase9.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase9.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 10

In [ ]:
df_final10 = df_final.loc[1750000:2000000]
df_final10

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1750000,1750001,Vention Mobile & Comp,Vention Kabel Toslink Optical Fiber Audio Cabl...,Rp28.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-k...,13319967
1750001,1750002,Vention Mobile & Comp,Vention Network Cable HDMI Extender LAN Cable ...,Rp618.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-n...,13319967
1750002,1750003,Vention Mobile & Comp,Vention Wireless HDMI Dongle Vidio Transmitter...,Rp839.000,1.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-w...,13319967
1750003,1750004,Vention Mobile & Comp,Vention Kabel HDMI 2.0 Cable Male to Male Upgr...,Rp219.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-k...,13319967
1750004,1750005,Vention Mobile & Comp,Vention Kabel HDMI 2.0 Cable Male to Male Upgr...,Rp58.000,5.0,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ventionacc/vention-k...,13319967
...,...,...,...,...,...,...,...,...
1999996,1999997,DIGAP Indonesia,Digap Car Charger Mobil Fast Charging PD QC 3....,Rp127.600,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
1999997,1999998,DIGAP Indonesia,Digap Tripod Kamera Aksi Tongsis Monopod Power...,Rp1.593.200,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
1999998,1999999,DIGAP Indonesia,Digap Lampu Proyektor LED Spotlight 10 Pattern...,Rp1.890.200,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
1999999,2000000,DIGAP Indonesia,Digap Kacamata Motor Motorcycle Safety Goggles...,Rp140.800,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724


In [ ]:
done_ids = set(f.replace(".jpg", "")for f in os.listdir(path_folder)if f.endswith(".jpg"))
df_todo = df_final10[~df_final10["ID_Product"].astype(str).isin(done_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures
        time.sleep(1)

100%|██████████| 242092/242092 [4:52:32<00:00, 13.79product/s, batch=485/485, Success: =228462, Fail: =13630]


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final10[~df_final10['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1750170,1750171,Hypermart Cianjur,GLICO POCKY CRUSHED BLUEBERRY YOGURT 38 GR - H...,Rp12.830,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/hypermartcianjur/gli...,10967576
1750171,1750172,Hypermart Cianjur,MISTER POTATO GHOST PAPER SEAWEED 40 GR - Hype...,Rp14.970,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/hypermartcianjur/mis...,10967576
1750174,1750175,Hypermart Cianjur,MISTER POTATO GHOST PAPER ORI 40 GR - Hypermart,Rp14.970,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/hypermartcianjur/mis...,10967576
1750188,1750189,Hypermart Cianjur,KODOMO BABY WIPES HAND AND MOUTH 50'S - Hypermart,Rp18.180,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/hypermartcianjur/kod...,10967576
1750210,1750211,Hypermart Cianjur,ROYALE PREMIUM SPRING BLOSSOM BOTOL 600 ML - H...,Rp18.180,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/hypermartcianjur/roy...,10967576
...,...,...,...,...,...,...,...,...
1999578,1999579,DIGAP Indonesia,Digap Electronic Load Tester Digital Bluetooth...,Rp1.849.300,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
1999624,1999625,DIGAP Indonesia,Digap Alat Set Perkakas Palu Obeng Cutter Mete...,Rp165.700,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
1999705,1999706,DIGAP Indonesia,Digap Pemutar Musik Portable MP3 Player 16GB S...,Rp126.100,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
1999739,1999740,DIGAP Indonesia,Digap Holder Gagang Sepeda Bike Bracket Mount ...,Rp40.300,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724


In [ ]:
available_ids   = []
unavailable_ids = []
write_lock      = threading.Lock()
thread_count     = 20

def check_image_available(row):
    session      = requests.Session()
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    product_id   = str(row["ID_Product"])
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = session.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        status   = "ada" if products else "kosong"
    except:
        status   = "kosong"

    with write_lock:
        if status == "ada":
            available_ids.append(product_id)
        else:
            unavailable_ids.append(product_id)

rows  = error_data.to_dict("records")
total = len(rows)

with ThreadPoolExecutor(max_workers=thread_count) as executor:
    futures = {executor.submit(check_image_available, row): row for row in rows}
    with tqdm(total=total, unit="produk") as pbar:
        for future in as_completed(futures):
            future.result()
            pbar.update(1)

print(f"Total Available Images   : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 13630/13630 [05:44<00:00, 39.59produk/s]

Total Available Images   : 1,229 (9.0%)
Total Unavailable Images : 12,401 (91.0%)


In [ ]:
available_ids[:5]

['1755904', '1758783', '1759014', '1760976', '1761681']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1755903,1755904,Second Step Coffee,Grinder Electric Portable Usb Charge Alat Peng...,Rp219.900,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/secondstepcoffee/gri...,12892522
1758782,1758783,Citra Furniture Official,Citra Furniture Spring Bed Big Koil Posture Pl...,Rp9.870.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/citra-furniture-offi...,7495143184639036342
1759013,1759014,Tadi Pagi Coffee Roastery,Powder Matcha Latte Bubuk La Terra 1 kg,Rp168.750,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/tadipagiroastery/pow...,6862321
1760975,1760976,Jane Frey Clothing_NEW,"JaneFrey K013 Celana Jogger Anak, Pakaian Ana...",Rp57.600,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/jane-frey-clothing/j...,7494489646279919761
1761500,1761501,Tagetto Coffee Roastery_NEW,BIJI KOPI ESPRESSO FLORES PREMIUM 500GR - ARAB...,Rp154.999,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/tagettocoffee/biji-k...,1143055
...,...,...,...,...,...,...,...,...
1999210,1999211,DIGAP Indonesia,Digap Stiker Vinyl Carbon Fiber Mobil Car Wrap...,Rp40.300,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
1999313,1999314,DIGAP Indonesia,Digap Kemoceng Mobil Pembersih Debu Microfiber...,Rp31.800,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
1999320,1999321,DIGAP Indonesia,Digap Rubber Strip Dekorasi Pintu Mobil Anti C...,Rp52.200,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
1999322,1999323,DIGAP Indonesia,Digap Flexible Camera Mount For GoPro DJI Acti...,Rp336.300,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 1229/1229 [01:23<00:00, 14.79product/s, batch=3/3, Success: =1219, Fail: =10]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-1*error_count:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 6/6 [00:00<00:00,  6.14product/s, batch=1/1, Success: =0, Fail: =6]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final10[~df_final10["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final10)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 250001
Total Images in Folder: 237594
Missing Images in Folder: 12407
Total Images & Missing Images in Folder 250001


In [ ]:
unavailable_ids[:5]

['1750298', '1750171', '1750329', '1750175', '1750283']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
1750170,1750171,Hypermart Cianjur,GLICO POCKY CRUSHED BLUEBERRY YOGURT 38 GR - H...,Rp12.830,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/hypermartcianjur/gli...,10967576
1750171,1750172,Hypermart Cianjur,MISTER POTATO GHOST PAPER SEAWEED 40 GR - Hype...,Rp14.970,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/hypermartcianjur/mis...,10967576
1750174,1750175,Hypermart Cianjur,MISTER POTATO GHOST PAPER ORI 40 GR - Hypermart,Rp14.970,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/hypermartcianjur/mis...,10967576
1750188,1750189,Hypermart Cianjur,KODOMO BABY WIPES HAND AND MOUTH 50'S - Hypermart,Rp18.180,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/hypermartcianjur/kod...,10967576
1750210,1750211,Hypermart Cianjur,ROYALE PREMIUM SPRING BLOSSOM BOTOL 600 ML - H...,Rp18.180,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/hypermartcianjur/roy...,10967576


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase10.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase10.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase10.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 11

In [ ]:
df_final11 = df_final.loc[2000000:2250000]
df_final11

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2000000,2000001,DIGAP Indonesia,Digap Lampu COB LED Strip Fleksibel 320 LED 24...,Rp277.200,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000001,2000002,DIGAP Indonesia,Digap Lampu Proyektor Disco RGB Animation Lase...,Rp3.775.900,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000002,2000003,DIGAP Indonesia,Digap Lensa Tele Smartphone Monocular Telephot...,Rp2.008.600,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000003,2000004,DIGAP Indonesia,Digap Lampu Depan Sepeda LED T6 USB Rechargeab...,Rp120.100,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000004,2000005,DIGAP Indonesia,Digap Printer Foto Smartphone Portable Bluetoo...,Rp3.777.400,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
...,...,...,...,...,...,...,...,...
2249996,2249997,Formiana,HAVIN Celemek Apron Dapur YY 7902 Bahan Serat,Rp41.796,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/havin-celem...,11098896
2249997,2249998,Formiana,HAVIN TALENAN BULAT TEBAL + HANDLE 36 CM,Rp116.532,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/havin-talen...,11098896
2249998,2249999,Formiana,HAVIN Cutlery Set BWJ068 Your Choice Big,Rp88.884,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/havin-cutle...,11098896
2249999,2250000,Formiana,HAVIN CUTLERY SET AB-551 small,Rp79.704,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/havin-cutle...,11098896


In [ ]:
df_todo = df_final11.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|█████████▉| 249934/250001 [2:37:40<00:02, 30.17product/s, batch=500/501, Success: =234779, Fail: =15155]

In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final11[~df_final11['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2000004,2000005,DIGAP Indonesia,Digap Printer Foto Smartphone Portable Bluetoo...,Rp3.777.400,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000099,2000100,DIGAP Indonesia,Digap Tatakan Kompor Induksi Heat Transfer Pla...,Rp265.900,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000362,2000363,DIGAP Indonesia,Digap Tempat Kuas Make Up Brush Storage Rotati...,Rp184.200,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000459,2000460,DIGAP Indonesia,Digap Korset Pembentuk Badan Pria Abdominal Wa...,Rp132.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000466,2000467,DIGAP Indonesia,Digap Bangku Dapur Kamar Mandi Kursi Jongkok M...,Rp159.100,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
...,...,...,...,...,...,...,...,...
2249986,2249987,Formiana,Handuk Motif Bordir Garis Jumbo,Rp45.252,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/handuk-moti...,11098896
2249988,2249989,Formiana,"HANDUK BT 32209 67,5 x 135 CM",Rp58.968,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/handuk-bt-3...,11098896
2249990,2249991,Formiana,Gelas Estetik Preston Milk Frothing Pitcher Hi...,Rp81.972,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/gelas-estet...,11098896
2249992,2249993,Formiana,MILK FROTHING PITCHER 19 OZ 550 ML,Rp76.248,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/milk-frothi...,11098896


In [ ]:
available_ids   = []
unavailable_ids = []
write_lock      = threading.Lock()
thread_count     = 20

def check_image_available(row):
    session      = requests.Session()
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    product_id   = str(row["ID_Product"])
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = session.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        status   = "ada" if products else "kosong"
    except:
        status   = "kosong"

    with write_lock:
        if status == "ada":
            available_ids.append(product_id)
        else:
            unavailable_ids.append(product_id)

rows  = error_data.to_dict("records")
total = len(rows)

with ThreadPoolExecutor(max_workers=thread_count) as executor:
    futures = {executor.submit(check_image_available, row): row for row in rows}
    with tqdm(total=total, unit="produk") as pbar:
        for future in as_completed(futures):
            future.result()
            pbar.update(1)

print(f"Total Available Images   : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 15169/15169 [03:39<00:00, 69.13produk/s]

Total Available Images   : 1,223 (8.1%)
Total Unavailable Images : 13,946 (91.9%)


In [ ]:
available_ids[:5]

['2001224', '2000986', '2001642', '2001897', '2001635']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2000985,2000986,DIGAP Indonesia,Digap Rak Gantungan Kitchen Utensil Peralatan ...,Rp136.400,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2001223,2001224,DIGAP Indonesia,Digap Lampu Outbow LED Ceiling Light Bright an...,Rp67.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2001445,2001446,DIGAP Indonesia,Digap Hanger Gantungan Baju Susun Retracable T...,Rp63.500,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2001447,2001448,DIGAP Indonesia,Digap Rak Monitor TV Top Shelf Display Screen ...,Rp75.200,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2001634,2001635,DIGAP Indonesia,Digap Rak Gantungan Hook Wall Hanger Organizer...,Rp24.600,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
...,...,...,...,...,...,...,...,...
2247200,2247201,Fitwear Indonesia,Fitwear- Jacket olahraga wanita OLIVIA CURVE 1...,Rp129.999,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fitwearofficial/fitw...,5696005
2248665,2248666,Fastbikes Indonesia,FASTBIKES - BREKET KLEMAN SPAKBOR COVER SPAKBO...,Rp89.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fastbikesid/fastbike...,15515685
2249221,2249222,Funnycook.Indonesia,[LIVE FENIHOME] Termos 1000ml dengan Gagang Ka...,Rp279.000,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/funnycookindonesia-9...,7495475007472044293
2249311,2249312,Funnycook.Indonesia,[YOURIDEALHOME] Funnycook Big Belly Pot Penggo...,Rp179.000,4.3,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/funnycookindonesia-9...,7495475007472044293


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 1223/1223 [00:43<00:00, 28.44product/s, batch=3/3, Success: =1219, Fail: =4]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-1*error_count:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 4/4 [00:00<00:00,  6.98product/s, batch=1/1, Success: =0, Fail: =4]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final11[~df_final11["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final11)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 250001
Total Images in Folder: 236051
Missing Images in Folder: 13950
Total Images & Missing Images in Folder 250001


In [ ]:
unavailable_ids[:5]

['2001131', '2000100', '2000363', '2000996', '2000467']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2000004,2000005,DIGAP Indonesia,Digap Printer Foto Smartphone Portable Bluetoo...,Rp3.777.400,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000099,2000100,DIGAP Indonesia,Digap Tatakan Kompor Induksi Heat Transfer Pla...,Rp265.900,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000362,2000363,DIGAP Indonesia,Digap Tempat Kuas Make Up Brush Storage Rotati...,Rp184.200,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000459,2000460,DIGAP Indonesia,Digap Korset Pembentuk Badan Pria Abdominal Wa...,Rp132.000,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724
2000466,2000467,DIGAP Indonesia,Digap Bangku Dapur Kamar Mandi Kursi Jongkok M...,Rp159.100,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/digap-indonesia/diga...,3514724


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase11.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase11.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase11.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 12

In [ ]:
df_final12 = df_final.loc[2250000:2750000]
df_final12

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2250000,2250001,Formiana,Kompor Gas Portable 3616 GSF 2 in 1 Gas Kaleng...,Rp132.192,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/kompor-gas-...,11098896
2250001,2250002,Formiana,Kotak Makan Lunch Box 1100 ml KH0048,Rp35.964,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/kotak-makan...,11098896
2250002,2250003,Formiana,Nampan Baki Stainless Stell tebal 32x22x4,Rp14.148,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/nampan-baki...,11098896
2250003,2250004,Formiana,Sealware Segi Pamelo Lemony Ukuran L Large Wad...,Rp18.684,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/sealware-se...,11098896
2250004,2250005,Formiana,FM - Mangkok Makan Plastik 075 Hitam Mercury D...,Rp6.048,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/fm-mangkok-...,11098896
...,...,...,...,...,...,...,...,...
2749996,2749936,Devil Store Indonesia,STARTRC Drone Landing Gear for DJI Flip Suppor...,Rp120.000,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/devil-store/startrc-...,1939474
2749997,2749937,Devil Store Indonesia,STARTRC DJI Accessories Protector Propeller Gu...,Rp120.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/startrc-...,1939474
2749998,2749938,Devil Store Indonesia,STARTRC Gimbal Protector Cover for DJI Flip Le...,Rp99.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/startrc-...,1939474
2749999,2749939,Devil Store Indonesia,Sunnylife Aluminum Alloy Control Sticks Storab...,Rp50.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/sunnylif...,1939474


In [ ]:
df_todo = df_final12.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 500001/500001 [9:09:39<00:00, 15.16product/s, batch=1001/1001, Success: =472430, Fail: =27571]


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final12[~df_final12['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2250002,2250003,Formiana,Nampan Baki Stainless Stell tebal 32x22x4,Rp14.148,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/nampan-baki...,11098896
2250003,2250004,Formiana,Sealware Segi Pamelo Lemony Ukuran L Large Wad...,Rp18.684,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/sealware-se...,11098896
2250005,2250006,Formiana,Dispenser Air 10Lt Plastik Motif Kristal Ikan ...,Rp167.292,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/dispenser-a...,11098896
2250008,2250009,Formiana,Gelas Minum Jerami 163 GSF / Gelas Plastik BPA...,Rp4.860,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/gelas-minum...,11098896
2250011,2250012,Formiana,Botol Minum Mixue Tumbler Viral Snow King,Rp12.960,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/botol-minum...,11098896
...,...,...,...,...,...,...,...,...
2749335,2749275,Neohaus Indonesia,"Swiss Military Solio Travel Luggage 18"" & 26"" ...",Rp2.980.000,4.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/neohaus/swiss-milita...,489109
2749752,2749692,IKUNKA Official Store,IKUNKA Sendal Cewek Sandal Flat Wanita Slip On...,Rp161.900,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ikunka-store/ikunka-...,7494223243088790902
2749771,2749711,IKUNKA Official Store,IKUNKA Ella Bow Flatshoes Basic Flat Shoes Wan...,Rp192.500,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/ikunka-store/ikunka-...,7494223243088790902
2749847,2749787,Devil Store Indonesia,Sunnylife Stretchable Backpack Clip Magnetic M...,Rp275.000,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/devil-store/sunnylif...,1939474


In [ ]:
available_ids   = []
unavailable_ids = []
write_lock      = threading.Lock()
thread_count     = 20

def check_image_available(row):
    session      = requests.Session()
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    product_id   = str(row["ID_Product"])
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = session.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        status   = "ada" if products else "kosong"
    except:
        status   = "kosong"

    with write_lock:
        if status == "ada":
            available_ids.append(product_id)
        else:
            unavailable_ids.append(product_id)

rows  = error_data.to_dict("records")
total = len(rows)

with ThreadPoolExecutor(max_workers=thread_count) as executor:
    futures = {executor.submit(check_image_available, row): row for row in rows}
    with tqdm(total=total, unit="produk") as pbar:
        for future in as_completed(futures):
            future.result()
            pbar.update(1)

print(f"Total Available Images   : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 27512/27512 [11:26<00:00, 40.06produk/s]

Total Available Images   : 7,216 (26.2%)
Total Unavailable Images : 20,296 (73.8%)


In [ ]:
available_ids[:5]

['2251470', '2252067', '2252066', '2252380', '2252386']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2251469,2251470,Apotek Filantropis by GoApotik,ROHTO EYE DROPS ISI 7 ML BOTOL,Rp16.886,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apotek-filantropis-b...,16186659
2252065,2252066,Apotek Family Farma Surabaya,ACETYLCYSTEINE MULIA FARMA 200 MG STRIP 10 KAPSUL,Rp12.179,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-family/acetyl...,12599143
2252066,2252067,Apotek Family Farma Surabaya,FORTIBOOST IMMUNO STRIP 6 KAPLET,Rp51.145,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-family/fortib...,12599143
2252379,2252380,FISHINGIN,"TEGEK Oregon Gelatik KOLONG BISA COD 180, 210,...",Rp71.725,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fishingin-24/tegek-o...,7494083655918584868
2252385,2252386,FISHINGIN,Joran Pancing Kolam Action Medium Indonesian P...,Rp90.450,4.5,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/fishingin-24/joran-p...,7494083655918584868
...,...,...,...,...,...,...,...,...
2745971,2745911,INTI MEDIKA STORE,ONEMED - Baju Jaga OK LENGAN PENDEK Set Perawa...,Rp150.000,1.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/intimedikabdg/onemed...,5240450
2745980,2745920,INTI MEDIKA STORE,ONEMED - Baju Pasien PATIENT GOWN SOFTIE One Med,Rp23.000,4.5,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/intimedikabdg/onemed...,5240450
2746552,2746492,INTI MEDIKA STORE,ONEMED - Baju APD Medis Putih 1412 | APD Baju ...,Rp99.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/intimedikabdg/onemed...,5240450
2746725,2746665,INTI MEDIKA STORE,CETAPHIL - Daily Facial Moisturizer SPF 15/PA+...,Rp208.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/intimedikabdg/cetaph...,5240450


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 7216/7216 [07:45<00:00, 15.50product/s, batch=15/15, Success: =7202, Fail: =14]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-1*error_count:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 14/14 [00:05<00:00,  2.34product/s, batch=1/1, Success: =2, Fail: =12]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final12[~df_final12["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final12)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 500001
Total Images in Folder: 479632
Missing Images in Folder: 20308
Total Images & Missing Images in Folder 499940


In [ ]:
unavailable_ids[:5]

['2250004', '2250033', '2250021', '2250023', '2250012']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2250002,2250003,Formiana,Nampan Baki Stainless Stell tebal 32x22x4,Rp14.148,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/nampan-baki...,11098896
2250003,2250004,Formiana,Sealware Segi Pamelo Lemony Ukuran L Large Wad...,Rp18.684,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/sealware-se...,11098896
2250005,2250006,Formiana,Dispenser Air 10Lt Plastik Motif Kristal Ikan ...,Rp167.292,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/dispenser-a...,11098896
2250008,2250009,Formiana,Gelas Minum Jerami 163 GSF / Gelas Plastik BPA...,Rp4.860,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/gelas-minum...,11098896
2250011,2250012,Formiana,Botol Minum Mixue Tumbler Viral Snow King,Rp12.960,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/formiana/botol-minum...,11098896


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase12.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase12.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase12.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 13

In [ ]:
df_final13 = df_final.loc[2750000:3250000]
df_final13

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2750000,2749940,Devil Store Indonesia,TELESIN Magnetic Motorcycle Mobile Phone Holde...,Rp275.000,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/devil-store/telesin-...,1939474
2750001,2749941,Devil Store Indonesia,TELESIN Smartphone Selfie Stick Retractable Po...,Rp275.000,NaN,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/devil-store/telesin-...,1939474
2750002,2749942,Devil Store Indonesia,Startrc ND Filters set for DJI Air 3 ND8 / 16 ...,Rp599.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/startrc-...,1939474
2750003,2749943,Devil Store Indonesia,Alien Monster Impermanence Glove Sarung Tangan...,Rp999.000,4.9,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/devil-store/alien-mo...,1939474
2750004,2749944,Devil Store Indonesia,Sunnylife Colorful Stickers Protective Film Sc...,Rp135.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/sunnylif...,1939474
...,...,...,...,...,...,...,...,...
3249996,3249936,PROTY Official Store,PROTY Original (Bundle 6) Snack Anak Sehat Pro...,Rp132.574,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/proty-official-store...,15316634
3249997,3249937,PROTY Official Store,PROTY Ori(2)+Pedas(1) Snack Cemilan Berprotein...,Rp81.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/proty-official-store...,15316634
3249998,3249938,PROTY Official Store,PROTY Pedas (Bundle 3) Snack Cemilan Protein D...,Rp71.188,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/proty-official-store...,15316634
3249999,3249939,PROTY Official Store,PROTY Original (Bundle 3) Snack Keluarga Sehat...,Rp70.198,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/proty-official-store...,15316634


In [ ]:
df_todo = df_final13.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 500001/500001 [9:40:27<00:00, 14.36product/s, batch=1001/1001, Success: =480823, Fail: =19178]


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final13[~df_final13['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2750075,2750015,Devil Store Indonesia,Sunnylife 40m Waterproof Case Diving Osmo Acti...,Rp250.000,4.5,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/sunnylif...,1939474
2750089,2750029,Devil Store Indonesia,Ringke Silicone Drafting Compatible for apple ...,Rp169.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/ringke-s...,1939474
2750163,2750103,Devil Store Indonesia,STARTRC 4-Pack ND Filters Set for DJI Mini 4K/...,Rp475.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/startrc-...,1939474
2750211,2750151,Devil Store Indonesia,Startrc Ultra HD Wide Angle Lens Filter DJi Mi...,Rp399.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/startrc-...,1939474
2750213,2750153,Devil Store Indonesia,Sunnylife Multifunctional Carrying Case Tas Sh...,Rp575.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/sunnylif...,1939474
...,...,...,...,...,...,...,...,...
3249827,3249767,Vaporesso Official Store,VAPORESSO XROS 3 Pod Kit 1000mAh Vape Pod 100%...,Rp260.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/vaporesso/vaporesso-...,14079440
3249838,3249778,Vaporesso Official Store,VAPORESSO LUXE XR MAX Mod Kit 2800mAh 5ml Mod ...,Rp390.000,4.9,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/vaporesso/vaporesso-...,14079440
3249840,3249780,Vaporesso Official Store,VAPORESSO XROS 3 Pod Kit 1000mAh Vape Pod 100%...,Rp235.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/vaporesso/vaporesso-...,14079440
3249846,3249786,1Zpresso Official Store,1Zpresso J-Ultra Coffee Grinder - Penggiling B...,Rp3.750.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/1zpresso-official-st...,11533738


In [ ]:
available_ids   = []
unavailable_ids = []
write_lock      = threading.Lock()
thread_count     = 20

def check_image_available(row):
    session      = requests.Session()
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    product_id   = str(row["ID_Product"])
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = session.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        status   = "ada" if products else "kosong"
    except:
        status   = "kosong"

    with write_lock:
        if status == "ada":
            available_ids.append(product_id)
        else:
            unavailable_ids.append(product_id)

rows  = error_data.to_dict("records")
total = len(rows)

with ThreadPoolExecutor(max_workers=thread_count) as executor:
    futures = {executor.submit(check_image_available, row): row for row in rows}
    with tqdm(total=total, unit="produk") as pbar:
        for future in as_completed(futures):
            future.result()
            pbar.update(1)

print(f"Total Available Images   : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 19178/19178 [07:52<00:00, 40.61produk/s]

Total Available Images   : 2,231 (11.6%)
Total Unavailable Images : 16,947 (88.4%)


In [ ]:
available_ids[:5]

['2750204', '2750302', '2750584', '2750015', '2750384']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2750075,2750015,Devil Store Indonesia,Sunnylife 40m Waterproof Case Diving Osmo Acti...,Rp250.000,4.5,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/sunnylif...,1939474
2750237,2750177,Devil Store Indonesia,Ringke Fast Charging Basic Braided Nylon Cable...,Rp199.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/ringke-f...,1939474
2750264,2750204,Devil Store Indonesia,Sunnylife 40m Waterproof Case Diving for Osmo ...,Rp250.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/sunnylif...,1939474
2750362,2750302,Devil Store Indonesia,Sunnylife 40m Waterproof Case Diving Osmo Acti...,Rp250.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/sunnylif...,1939474
2750444,2750384,Devil Store Indonesia,Sunnylife 40m Waterproof Case Diving Osmo Acti...,Rp250.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/sunnylif...,1939474
...,...,...,...,...,...,...,...,...
3244585,3244525,Unilever Official Store,Rinso Washing Machine Cleaner Pembersih Mesin ...,Rp91.500,4.9,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/unilever-official-st...,1854168
3247567,3247507,RUMAHKU OFFICIAL,COOKER HOOD ARTUGO AX 6122 SB,Rp1.336.674,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/rumahku-official/coo...,2661479
3247762,3247702,RUMAHKU OFFICIAL,Bosch MUM6N20A1 Kitchen Machine / Stand Mixer ...,Rp5.049.000,4.9,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/rumahku-official/bos...,2661479
3249745,3249685,Samson Official Store,Samson CM20P Gooseneck Podium Microphone,Rp1.336.500,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/samson-official-stor...,13932693


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 2231/2231 [03:07<00:00, 11.91product/s, batch=5/5, Success: =2220, Fail: =11]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-1*error_count:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 11/11 [00:01<00:00, 10.34product/s, batch=1/1, Success: =0, Fail: =11]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final13[~df_final13["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final13)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 500001
Total Images in Folder: 483043
Missing Images in Folder: 16958
Total Images & Missing Images in Folder 500001


In [ ]:
unavailable_ids[:5]

['2750151', '2754360', '2754489', '2750103', '2752362']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
2750089,2750029,Devil Store Indonesia,Ringke Silicone Drafting Compatible for apple ...,Rp169.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/ringke-s...,1939474
2750163,2750103,Devil Store Indonesia,STARTRC 4-Pack ND Filters Set for DJI Mini 4K/...,Rp475.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/startrc-...,1939474
2750211,2750151,Devil Store Indonesia,Startrc Ultra HD Wide Angle Lens Filter DJi Mi...,Rp399.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/startrc-...,1939474
2750213,2750153,Devil Store Indonesia,Sunnylife Multifunctional Carrying Case Tas Sh...,Rp575.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/sunnylif...,1939474
2750283,2750223,Devil Store Indonesia,Startrc Portable Protection Hand Storage Bag D...,Rp225.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/devil-store/startrc-...,1939474


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase13.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase13.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase13.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## phase 14

In [ ]:
drive.mount('/content/drive')
zip_path = "/content/drive/MyDrive/Data_Images1/images_product.zip"
destination_folder = "/content/images_product"
os.makedirs(destination_folder, exist_ok=True)
!unzip -q "{zip_path}" -d /content/

Mounted at /content/drive


In [ ]:
df_final14 = df_final.loc[3250000:]
df_final14

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
3250000,3249940,WELLEN Official,WELLEN Bed SET 100% Cellulose Fiber - Oriental,Rp6.957.500,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-bed-se...,8164234
3250001,3249941,WELLEN Official,WELLEN Bed SET 100% Cellulose Fiber - Green Fo...,Rp6.957.500,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-bed-se...,8164234
3250002,3249942,WELLEN Official,WELLEN PREMIUM Cotton Sateen Bed Set / Bed Cov...,Rp4.785.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-premiu...,8164234
3250003,3249943,WELLEN Official,WELLEN Premium Pillow Protector,Rp422.500,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-premiu...,8164234
3250004,3249944,WELLEN Official,WELLEN Bed SET 100% Cellulose Fiber - Golden Leaf,Rp7.210.500,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-bed-se...,8164234
...,...,...,...,...,...,...,...,...
3614808,3614748,Apotek Ubay Farma by GoApotik,COMBIVENT UDV CAIRAN INHALASI 2.5 ML PACK 10 VIAL,Rp66.526,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614809,3614749,Apotek Ubay Farma by GoApotik,COMBIVENT UDV CAIRAN INHALASI 2.5 ML BOX 20 VIAL,Rp114.908,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614810,3614750,Apotek Ubay Farma by GoApotik,FLUIMUCIL 100 MG/ML CAIRAN INHALASI 3 ML AMPUL,Rp23.138,5.0,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614811,3614751,Apotek Ubay Farma by GoApotik,FLUIMUCIL 100 MG/ML CAIRAN INHALASI 3 ML BOX 5...,Rp109.595,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250


In [ ]:
df_todo = df_final14.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 364813/364813 [6:07:48<00:00, 16.53product/s, batch=730/730, Success: =351995, Fail: =12818]


In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final14[~df_final14['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
3250010,3249950,WELLEN Official,WELLEN PREMIUM Cotton Sateen Bed Set & Bed Cov...,Rp4.785.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-premiu...,8164234
3250012,3249952,WELLEN Official,WELLEN JACQUARD Bed Sheet / Bed cover 100% Bam...,Rp6.875.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-jacqua...,8164234
3250019,3249959,WELLEN Official,WELLEN Bed Set / Cover 100% Bambu Cellulose Fi...,Rp6.957.500,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-bed-se...,8164234
3250022,3249962,WELLEN Official,WELLEN Bed Set / Cover 100% Bamboo Cellulose F...,Rp6.957.500,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-bed-se...,8164234
3250026,3249966,WELLEN Official,WELLEN Bed Set / Cover 100% Bambu Cellulose Fi...,Rp6.957.500,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-bed-se...,8164234
...,...,...,...,...,...,...,...,...
3614508,3614448,Apotek Ubay Farma by GoApotik,ERPHAFLAM 50 MG BOX 50 TABLET,Rp23.747,5.0,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614532,3614472,Apotek Ubay Farma by GoApotik,CESTER PER BOX ISI 30 KAPSUL,Rp101.070,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614558,3614498,Apotek Ubay Farma by GoApotik,RANTIN 150 MG STRIP 10 TABLET,Rp73.063,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614693,3614633,Apotek Ubay Farma by GoApotik,SURBEX Z STRIP ISI 6 TABLET,Rp29.225,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250


In [ ]:
available_ids   = []
unavailable_ids = []
write_lock      = threading.Lock()
thread_count     = 20

def check_image_available(row):
    session      = requests.Session()
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    product_id   = str(row["ID_Product"])
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = session.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        status   = "ada" if products else "kosong"
    except:
        status   = "kosong"

    with write_lock:
        if status == "ada":
            available_ids.append(product_id)
        else:
            unavailable_ids.append(product_id)

rows  = error_data.to_dict("records")
total = len(rows)

with ThreadPoolExecutor(max_workers=thread_count) as executor:
    futures = {executor.submit(check_image_available, row): row for row in rows}
    with tqdm(total=total, unit="produk") as pbar:
        for future in as_completed(futures):
            future.result()
            pbar.update(1)

print(f"Total Available Images   : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 12818/12818 [04:44<00:00, 45.01produk/s]

Total Available Images   : 2,005 (15.6%)
Total Unavailable Images : 10,813 (84.4%)


In [ ]:
available_ids[:5]

['3250006', '3249984', '3249971', '3249959', '3249966']

In [ ]:
error_data[error_data["ID_Product"].astype(str).isin(available_ids)]

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
3250010,3249950,WELLEN Official,WELLEN PREMIUM Cotton Sateen Bed Set & Bed Cov...,Rp4.785.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-premiu...,8164234
3250012,3249952,WELLEN Official,WELLEN JACQUARD Bed Sheet / Bed cover 100% Bam...,Rp6.875.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-jacqua...,8164234
3250019,3249959,WELLEN Official,WELLEN Bed Set / Cover 100% Bambu Cellulose Fi...,Rp6.957.500,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-bed-se...,8164234
3250022,3249962,WELLEN Official,WELLEN Bed Set / Cover 100% Bamboo Cellulose F...,Rp6.957.500,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-bed-se...,8164234
3250026,3249966,WELLEN Official,WELLEN Bed Set / Cover 100% Bambu Cellulose Fi...,Rp6.957.500,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/wellen/wellen-bed-se...,8164234
...,...,...,...,...,...,...,...,...
3606974,3606914,Tennesy,TENNESY SIGN PLAKAT SET ISI 6 RUANG KANTOR RUM...,Rp59.400,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/tennesyhomedecoratio...,12859077
3613748,3613688,Apotek Ubay Farma by GoApotik,ASPILETS 80 MG BOX 100 TABLET,Rp97.416,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614045,3613985,Apotek Ubay Farma by GoApotik,POLYSILANE STRIP 8 TABLET,Rp10.961,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614159,3614099,Apotek Ubay Farma by GoApotik,ASPILETS 80 MG STRIP 10 TABLET,Rp103.506,5.0,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 2005/2005 [01:51<00:00, 17.99product/s, batch=5/5, Success: =1900, Fail: =105]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-1*error_count:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 104/104 [00:02<00:00, 51.12product/s, batch=1/1, Success: =0, Fail: =104]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final14[~df_final14["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final14)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 364813
Total Images in Folder: 353896
Missing Images in Folder: 10917
Total Images & Missing Images in Folder 364813


In [ ]:
unavailable_ids[:5]

['3250209', '3250208', '3250228', '3250244', '3250256']

In [ ]:
missing_id_product.head(5)

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
3250268,3250208,Mamilo Online Store,MAMILO Spons Mandi Jaring Shower Puff Bath Bulat,Rp5.900,4.8,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mamiloofficial/mamil...,10240759
3250269,3250209,Mamilo Online Store,MAMILO Piring Makan Plastik Small Plate Set 6 ...,Rp29.000,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/mamiloofficial/mamil...,10240759
3250288,3250228,Mamilo Online Store,MAMILO Rak Sepatu 4 Susun / Lemari Sepatu / Ra...,Rp149.000,4.7,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/mamiloofficial/mamil...,10240759
3250299,3250239,Mamilo Online Store,MAMILO Tempat Tissue Dispenser Plastik Organiz...,Rp17.000,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/mamiloofficial/mamil...,10240759
3250304,3250244,Mamilo Online Store,MAMILO Pel Lantai X Mop Kualitas TEBAL Pel Per...,Rp120.000,4.9,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/mamiloofficial/mamil...,10240759


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
with open('totally_no_iamges_during_phase14.json', 'w') as f:
    json.dump(missing_ids, f)
files.download('totally_no_iamges_during_phase14.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase14.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# cleaning unavailable images

In [ ]:
import pandas as pd
from google.colab import drive
import json
import os

drive.mount('/content/drive')
df_final = pd.read_csv("/content/drive/MyDrive/df_final.csv")
df_sid = pd.read_csv("/content/drive/MyDrive/Scrapping_list_shop_with_SID.csv")
df_final = df_final.merge(df_sid[['Nama Toko', 'SID']], left_on='Shop_Name', right_on='Nama Toko', how='left')
df_final.drop(columns=['Nama Toko'], inplace=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
json_data = {}
for i in range(1, 15):
    file_name = f'totally_no_iamges_during_phase{i}.json'
    file_path = f'/content/drive/MyDrive/{file_name}'
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            json_data[f'phase{i}'] = json.load(f)
print(f"{len(json_data)} JSON files.")

14 JSON files.


In [ ]:
all_id_product = []
for values in json_data.values():
    all_id_product.extend(values)
all_id_product = [str(i) for i in all_id_product]
len(all_id_product)

171194

In [ ]:
df_todo = df_final[df_final["ID_Product"].astype(str).isin(all_id_product)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 171193/171193 [1:22:28<00:00, 34.59product/s, batch=343/343, Success: =24182, Fail: =147011]


In [ ]:
df_final15 = df_final[df_final["ID_Product"].astype(str).isin(all_id_product)].copy()
existing_images = {f.split('.')[0] for f in os.listdir('images_product')}
error_data = df_final15[~df_final15['ID_Product'].astype(str).isin(existing_images)]
error_data

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
42,43,Apotek Sengeti Farma Sekernan,ALLOPURINOL 100MG 1 STRIP 10 TABLET (Gen HJ),Rp4.976,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
43,44,Apotek Sengeti Farma Sekernan,INTERHISTIN 50MG 1 STRIP ISI 10 TABLET,Rp16.813,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
44,45,Apotek Sengeti Farma Sekernan,IMODIUM 2MG 1 STRIP 10 TABLET,Rp143.783,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
45,46,Apotek Sengeti Farma Sekernan,FARMOTEN 25MG 1 STRIP 10 TABLET,Rp5.499,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
51,52,Apotek Sengeti Farma Sekernan,"HARNAL OCAS 0,4MG 1 BLISTER 10 TABLET",Rp161.068,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/apoteksengetifarmase...,17115861
...,...,...,...,...,...,...,...,...
3614508,3614448,Apotek Ubay Farma by GoApotik,ERPHAFLAM 50 MG BOX 50 TABLET,Rp23.747,5.0,https://p19-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614532,3614472,Apotek Ubay Farma by GoApotik,CESTER PER BOX ISI 30 KAPSUL,Rp101.070,NaN,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614558,3614498,Apotek Ubay Farma by GoApotik,RANTIN 150 MG STRIP 10 TABLET,Rp73.063,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
3614693,3614633,Apotek Ubay Farma by GoApotik,SURBEX Z STRIP ISI 6 TABLET,Rp29.225,5.0,https://p16-images-common-sign-sg.tokopedia-st...,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250


In [ ]:
available_ids   = []
unavailable_ids = []
write_lock      = threading.Lock()
thread_count     = 20

def check_image_available(row):
    session      = requests.Session()
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    product_id   = str(row["ID_Product"])
    keyword      = ' '.join(str(product_name).split()[:3])

    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": sid, "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]

    try:
        res      = session.post('https://gql.tokopedia.com/graphql/ShopProducts', headers=headers, json=payload, timeout=15)
        products = res.json()[0]['data']['GetShopProduct']['data']
        status   = "ada" if products else "kosong"
    except:
        status   = "kosong"

    with write_lock:
        if status == "ada":
            available_ids.append(product_id)
        else:
            unavailable_ids.append(product_id)

rows  = error_data.to_dict("records")
total = len(rows)

with ThreadPoolExecutor(max_workers=thread_count) as executor:
    futures = {executor.submit(check_image_available, row): row for row in rows}
    with tqdm(total=total, unit="produk") as pbar:
        for future in as_completed(futures):
            future.result()
            pbar.update(1)

print(f"Total Available Images   : {len(available_ids):,} ({len(available_ids)/total*100:.1f}%)")
print(f"Total Unavailable Images : {len(unavailable_ids):,} ({len(unavailable_ids)/total*100:.1f}%)")

100%|██████████| 147011/147011 [50:23<00:00, 48.62produk/s]

Total Available Images   : 278 (0.2%)
Total Unavailable Images : 146,733 (99.8%)


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(available_ids)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 278/278 [00:20<00:00, 13.74product/s, batch=1/1, Success: =249, Fail: =29]


In [ ]:
df_todo = error_data[error_data["ID_Product"].astype(str).isin(error_list[-1*error_count:])].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 29/29 [00:06<00:00,  4.75product/s, batch=1/1, Success: =0, Fail: =29]


In [ ]:
existing_files = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df_final15[~df_final15["ID_Product"].astype(str).isin(existing_files)].copy()

print(f"Total Data: {len(df_final15)}")
print(f"Total Images in Folder: {len(existing_files)}")
print(f"Missing Images in Folder: {len(missing_id_product)}")
print(f"Total Images & Missing Images in Folder {len(missing_id_product)+len(existing_files)}")

Total Data: 171193
Total Images in Folder: 24431
Missing Images in Folder: 146762
Total Images & Missing Images in Folder 171193


In [ ]:
missing_ids = missing_id_product['ID_Product'].tolist()
file_path = '/content/drive/MyDrive/totally_no_iamges_during_phase15.json'

with open(file_path, 'w') as f:
    json.dump(missing_ids, f)

In [ ]:
drive.mount('/content/drive')
destination_path = "/content/drive/MyDrive/Data_Images1"
zip_name = "images_product_phase15.zip"
full_zip_path = os.path.join(destination_path, zip_name)
os.makedirs(destination_path, exist_ok=True)
!zip -rq "{full_zip_path}" images_product

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
len(missing_id_product)

146762

In [ ]:
df_finalized = df_final[~df_final["ID_Product"].astype(str).isin(missing_id_product)].copy()
df_finalized.head()

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product_Picture,URL_Product,SID
0,1,Awkward Brand,Kaos Polos Anak Katun Combed 30s Hijau Olive,Rp74.000,NaN,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...,14656547
1,2,Awkward Brand,Kaos Polos Anak Katun Combed 30s Biru Navy,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...,14656547
2,3,Awkward Brand,Kaos Polos Anak Katun Combed 30s Kuning Mustard,Rp74.000,NaN,https://p19-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...,14656547
3,4,Awkward Brand,Kaos Polos Anak Katun Combed 30s Merah Maroon,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...,14656547
4,5,Awkward Brand,Kaos Polos Anak Katun Combed 30s Abu-abu Misty,Rp74.000,5.0,https://p16-images-sign-sg.tokopedia-static.ne...,https://www.tokopedia.com/awkwardofficial/kaos...,14656547


In [ ]:
df_finalized.columns

Index(['ID_Product', 'Shop_Name', 'Product_Name', 'Price', 'Rating',
       'URL_Product_Picture', 'URL_Product', 'SID'],
      dtype='object')

In [ ]:
df_finalized = df_finalized.drop(columns=["URL_Product_Picture","SID"],axis=1)
save_path = "/content/drive/MyDrive/tokopedia_products.csv"
df_finalized.to_csv(save_path, index=False, encoding='utf-8-sig')

In [ ]:
df_finalized1 = pd.read_csv("tokopedia_products.csv")
print(df_finalized1.head())
print(df_finalized1.columns)
print(f"Total baris: {df_finalized1.shape[0]}")

   ID_Product      Shop_Name                                     Product_Name  \
0           1  Awkward Brand     Kaos Polos Anak Katun Combed 30s Hijau Olive   
1           2  Awkward Brand       Kaos Polos Anak Katun Combed 30s Biru Navy   
2           3  Awkward Brand  Kaos Polos Anak Katun Combed 30s Kuning Mustard   
3           4  Awkward Brand    Kaos Polos Anak Katun Combed 30s Merah Maroon   
4           5  Awkward Brand   Kaos Polos Anak Katun Combed 30s Abu-abu Misty   

      Price  Rating                                        URL_Product  
0  Rp74.000     NaN  https://www.tokopedia.com/awkwardofficial/kaos...  
1  Rp74.000     5.0  https://www.tokopedia.com/awkwardofficial/kaos...  
2  Rp74.000     NaN  https://www.tokopedia.com/awkwardofficial/kaos...  
3  Rp74.000     5.0  https://www.tokopedia.com/awkwardofficial/kaos...  
4  Rp74.000     5.0  https://www.tokopedia.com/awkwardofficial/kaos...  
Index(['ID_Product', 'Shop_Name', 'Product_Name', 'Price', 'Rating',
      